# End-to-End Steering Prediction for Self-Driving Cars

This notebook contains the exploratory modeling workflow behind the repository. Outputs have been cleared for version control; regenerate them by running the notebook after placing the dataset under `data/raw/`.

# End-to-End Steering Prediction for Self-Driving Cars

This notebook contains the exploratory modeling workflow behind the repository. Outputs have been cleared for version control; regenerate them by running the notebook after placing the dataset under `data/raw/`.

## Problem Statement and Motivation

The objective of this project is to build a model that predicts the steering angle of a simulated self-driving car directly from camera images. Traditional autonomous-driving pipelines often separate the problem into perception, lane detection, path planning, and control. In contrast, end-to-end learning attempts to learn the complete mapping from image pixels to steering command in one model.

This project is motivated by NVIDIA's end-to-end learning concept, where a convolutional neural network learns steering behavior from driving images. The Udacity simulator provides a safe environment for generating driving data and testing whether learned models can control a vehicle in autonomous mode.

The main machine-learning challenge is dataset imbalance. Most driving images correspond to straight-line motion, while sharp turns are rare. A model can achieve low average error by predicting near-zero steering for most frames, but such a model may fail in safety-critical cornering situations. For this reason, the project evaluates both overall accuracy and steering-region performance.

## Main Features Implemented

The notebook implements the following major features:

1. **Dataset validation and path correction** — verifies CSV structure, reconstructs valid image paths, and checks that all images are available.
2. **Exploratory data analysis** — visualizes representative camera images and quantifies steering-angle imbalance.
3. **Leakage-resistant splitting** — groups sequential frames before creating training, validation, and test splits.
4. **Side-camera sample expansion** — uses center, left, and right cameras with steering corrections to improve recovery behavior.
5. **Training-set balancing** — reduces over-representation of near-zero steering samples.
6. **Data augmentation** — applies translation, brightness variation, shadow augmentation, and horizontal flipping with label correction.
7. **Multiple model families** — trains and compares PilotNet, improved CNN, MobileNetV2, CLIP-based regression, and tuned PilotNet.
8. **Regional evaluation** — evaluates sharp-left, left, straight, right, and sharp-right steering regions separately.
9. **Hyperparameter tuning** — tunes PilotNet using macro regional validation MAE.
10. **Model export** — exports trained models to `.h5` format for simulator deployment.

## Design Framework / System Architecture

The complete system follows this design flow:

```text
Udacity simulator data
        ↓
driving_log.csv + IMG folder
        ↓
path validation and data cleaning
        ↓
steering-distribution analysis
        ↓
grouped train/validation/test split
        ↓
training-only balancing and augmentation
        ↓
model-specific preprocessing
        ↓
model training and validation
        ↓
test evaluation and regional analysis
        ↓
model export to .h5
        ↓
Socket.IO simulator deployment
```

During training, the model receives preprocessed images and learns to regress the steering angle. During simulator deployment, the simulator streams camera frames to a Python server. The server applies the same preprocessing, calls the trained model, and sends the predicted steering angle and throttle command back to the simulator.

## Model Versions and Reasons for Their Selection

| Model version | Reason for selection | Main purpose |
|---|---|---|
| **Original PilotNet** | Classical NVIDIA-style end-to-end CNN baseline | Establish a reference model trained from scratch |
| **Improved weighted CNN** | Tests whether regularization, Huber loss, and steering weights improve the baseline | Improve parameter efficiency and turn sensitivity |
| **Frozen MobileNetV2** | Evaluates transfer learning from a pretrained visual backbone | Test whether ImageNet visual features improve steering regression |
| **Fine-tuned MobileNetV2** | Tests whether adapting the upper backbone layers improves validation performance | Compare frozen versus partially adapted transfer learning |
| **CLIP encoder + regressor** | Adds a vision-language-pretrained visual representation | Test whether CLIP image embeddings are useful for steering prediction |
| **Tuned PilotNet** | Uses hyperparameter search guided by macro regional MAE | Improve balanced performance across straight and turning regions |

This model selection strategy compares a classical end-to-end CNN, a regularized CNN, a conventional transfer-learning model, a vision-language-pretrained encoder, and a task-specific tuned model.

## Setup, Runtime Requirements, and Usage Notes

The notebook was developed primarily in Google Colab. The expected setup is:

- Google Drive folder containing `driving_log.csv` and the `IMG/` directory.
- Python libraries: TensorFlow/Keras, NumPy, Pandas, OpenCV, Matplotlib, Scikit-learn, Pillow, PyTorch, and Hugging Face Transformers.
- Optional simulator deployment environment using WSL or Linux with TensorFlow, OpenCV, Flask, Eventlet, and legacy Socket.IO/Engine.IO support.

Important precautions:

- Do not change preprocessing between training and inference. PilotNet-style models require cropped, resized, YUV-normalized input of shape `(66, 200, 3)`.
- MobileNetV2 requires cropped RGB input resized to `(96, 160, 3)` with MobileNetV2 preprocessing.
- The CLIP regression head cannot directly accept simulator images; it requires a 512-dimensional CLIP image embedding.
- Only the training split is balanced and augmented. Validation and test splits remain unchanged for fair evaluation.
- The test set is evaluated only after model selection to avoid tuning on the test data.

### Project Setup and Reproducibility

Mount Google Drive, import the core scientific and machine-learning libraries, set random seeds, and inspect the Colab hardware environment. Deterministic settings make repeated runs more comparable, while the GPU check confirms whether TensorFlow can use an accelerator.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import random
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Enable deterministic operations where supported.
try:
    tf.config.experimental.enable_op_determinism()
    print("TensorFlow deterministic operations enabled.")
except Exception as error:
    print("Deterministic operations could not be fully enabled:")
    print(error)

print("\nPython version:")
print(sys.version)

print("\nPlatform:")
print(platform.platform())

print("\nTensorFlow version:")
print(tf.__version__)

print("\nOpenCV version:")
print(cv2.__version__)

print("\nPandas version:")
print(pd.__version__)

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print(f"\nGPU detected: {len(gpus)}")
    for gpu in gpus:
        print(gpu)

    # Prevent TensorFlow from allocating all GPU memory immediately.
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError as error:
            print(error)
else:
    print("\nNo GPU detected.")
    print("In Colab, select Runtime > Change runtime type > T4 GPU.")

print("\nGoogle Drive mounted successfully.")

### Locate the Dataset

Define the Google Drive project directory, inspect its contents, locate the driving log, and discover available image folders and files. This block prevents later failures caused by incorrect paths or unexpected folder names.


In [ ]:
from pathlib import Path
import pandas as pd
import os

# Main project directory on Google Drive
PROJECT_DIR = Path("/content/drive/MyDrive/ML_Python_Project")

print("Project directory:", PROJECT_DIR)
print("Directory exists:", PROJECT_DIR.exists())

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"Project directory was not found:\n{PROJECT_DIR}"
    )

print("\nContents of the project directory:")

for item in sorted(PROJECT_DIR.iterdir()):
    item_type = "Folder" if item.is_dir() else "File"
    print(f"{item_type:7s}: {item.name}")

csv_files = sorted(PROJECT_DIR.rglob("*.csv"))

print("\nCSV files found:")

if len(csv_files) == 0:
    raise FileNotFoundError(
        f"No CSV files were found inside:\n{PROJECT_DIR}"
    )

for index, csv_file in enumerate(csv_files):
    print(f"[{index}] {csv_file}")

# Prefer a file named driving_log.csv
matching_logs = [
    path for path in csv_files
    if path.name.lower() == "driving_log.csv"
]

if matching_logs:
    CSV_PATH = matching_logs[0]
else:
    CSV_PATH = csv_files[0]
    print(
        "\nWarning: driving_log.csv was not found by name."
        "\nThe first CSV file will be inspected instead."
    )

print("\nSelected CSV file:")
print(CSV_PATH)

possible_image_dirs = [
    path for path in PROJECT_DIR.rglob("*")
    if path.is_dir() and path.name.lower() in ["img", "images", "image"]
]

print("\nPossible image directories:")

if possible_image_dirs:
    for index, image_dir in enumerate(possible_image_dirs):
        print(f"[{index}] {image_dir}")
else:
    print("No folder named IMG, Images, or Image was found.")

image_extensions = {".jpg", ".jpeg", ".png", ".bmp"}

all_image_files = [
    path for path in PROJECT_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in image_extensions
]

print("\nTotal image files found:", len(all_image_files))

if all_image_files:
    print("\nFirst five image files:")
    for image_path in all_image_files[:5]:
        print(image_path)

print("\nFirst five raw lines of the CSV:\n")

with open(CSV_PATH, "r", encoding="utf-8-sig", errors="replace") as file:
    for _ in range(5):
        line = file.readline()

        if not line:
            break

        print(line.strip())

### Load and Clean the Driving Log

Read the CSV while automatically handling files with or without a header row. Camera paths are cleaned, numerical columns are converted to numeric types, and rows missing essential steering or camera information are removed.


In [ ]:
import pandas as pd
import numpy as np

EXPECTED_COLUMNS = [
    "center",
    "left",
    "right",
    "steering",
    "throttle",
    "brake",
    "speed"
]

data_initial = pd.read_csv(
    CSV_PATH,
    skipinitialspace=True
)

print("Columns detected during the first read:")
print(data_initial.columns.tolist())

normalized_columns = [
    str(column).strip().lower()
    for column in data_initial.columns
]

contains_expected_header = (
    "center" in normalized_columns
    and "steering" in normalized_columns
)

if contains_expected_header:
    data = data_initial.copy()

    # Normalize column names
    data.columns = normalized_columns

    print("\nCSV header detected.")
else:
    print("\nNo valid header detected. Reloading with assigned column names.")

    data = pd.read_csv(
        CSV_PATH,
        names=EXPECTED_COLUMNS,
        skipinitialspace=True
    )

missing_columns = [
    column for column in EXPECTED_COLUMNS
    if column not in data.columns
]

if missing_columns:
    print("\nMissing expected columns:")
    print(missing_columns)

    raise ValueError(
        "The CSV structure does not match the expected "
        "Udacity driving log format."
    )

data = data[EXPECTED_COLUMNS].copy()

camera_columns = ["center", "left", "right"]

for column in camera_columns:
    data[column] = (
        data[column]
        .astype(str)
        .str.strip()
    )

numeric_columns = [
    "steering",
    "throttle",
    "brake",
    "speed"
]

for column in numeric_columns:
    data[column] = pd.to_numeric(
        data[column],
        errors="coerce"
    )

rows_before_cleaning = len(data)

data = data.dropna(
    subset=[
        "center",
        "left",
        "right",
        "steering"
    ]
).reset_index(drop=True)

rows_after_cleaning = len(data)

print("\nDataset loaded successfully.")
print("Rows before cleaning:", rows_before_cleaning)
print("Rows after cleaning :", rows_after_cleaning)
print("Rows removed        :", rows_before_cleaning - rows_after_cleaning)

print("\nDataset shape:")
print(data.shape)

print("\nFirst five rows:")
display(data.head())

print("\nData types:")
print(data.dtypes)

print("\nMissing values:")
print(data.isna().sum())

print("\nSteering statistics:")
display(data["steering"].describe())

### Resolve Camera Image Paths

The simulator log may contain absolute paths from another operating system. This block extracts each filename, matches it to an image in the project directory, verifies all three camera views, and removes any unresolved records.


In [ ]:
from pathlib import Path
import os

image_lookup = {}

for image_path in all_image_files:
    filename = image_path.name

    # Store the first matching file for each filename
    if filename not in image_lookup:
        image_lookup[filename] = image_path

print("Unique image filenames indexed:", len(image_lookup))

def resolve_image_path(csv_path):
    """
    Convert a path stored in driving_log.csv into an existing
    path inside the Google Drive project directory.
    """

    csv_path = str(csv_path).strip()

    # Replace Windows separators so Path can extract the filename.
    normalized_path = csv_path.replace("\\", "/")

    filename = Path(normalized_path).name

    resolved_path = image_lookup.get(filename)

    if resolved_path is None:
        return None

    return str(resolved_path)


for camera in ["center", "left", "right"]:
    data[f"{camera}_path"] = data[camera].apply(resolve_image_path)

for camera in ["center", "left", "right"]:
    path_column = f"{camera}_path"

    missing_count = data[path_column].isna().sum()

    print(
        f"{camera.capitalize():6s} images missing: "
        f"{missing_count} / {len(data)}"
    )

# A row is valid only if all three camera images exist.
valid_image_mask = (
    data["center_path"].notna()
    & data["left_path"].notna()
    & data["right_path"].notna()
)

invalid_rows = data.loc[~valid_image_mask].copy()

print("\nRows with at least one missing image:", len(invalid_rows))

if len(invalid_rows) > 0:
    print("\nExamples of unresolved paths:")
    display(
        invalid_rows[
            [
                "center",
                "left",
                "right",
                "center_path",
                "left_path",
                "right_path"
            ]
        ].head()
    )

# Keep only valid rows
data = data.loc[valid_image_mask].reset_index(drop=True)

print("\nRows remaining after image-path validation:", len(data))

print("\nResolved paths from the first row:")
print("Center:", data.loc[0, "center_path"])
print("Left  :", data.loc[0, "left_path"])
print("Right :", data.loc[0, "right_path"])

### Visual Inspection of Camera Data

Display representative left-turn, straight-driving, and right-turn records using all three cameras. This qualitative check confirms that the images are readable, correctly oriented, synchronized, and paired with reasonable steering labels.


In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

# Choose representative rows:
# one strong left turn, one near-straight sample, one strong right turn
left_index = data["steering"].idxmin()
straight_index = data["steering"].abs().idxmin()
right_index = data["steering"].idxmax()

representative_indices = {
    "Strong left turn": left_index,
    "Near straight": straight_index,
    "Strong right turn": right_index
}

def read_rgb_image(image_path):
    """
    Read an image using OpenCV and convert BGR to RGB.
    """
    image = cv2.imread(str(image_path))

    if image is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")

    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


for sample_name, row_index in representative_indices.items():

    row = data.loc[row_index]

    center_image = read_rgb_image(row["center_path"])
    left_image = read_rgb_image(row["left_path"])
    right_image = read_rgb_image(row["right_path"])

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].imshow(left_image)
    axes[0].set_title("Left camera")
    axes[0].axis("off")

    axes[1].imshow(center_image)
    axes[1].set_title(
        f"Center camera\nSteering = {row['steering']:.4f}"
    )
    axes[1].axis("off")

    axes[2].imshow(right_image)
    axes[2].set_title("Right camera")
    axes[2].axis("off")

    fig.suptitle(sample_name, fontsize=14)
    plt.tight_layout()
    plt.show()

    print(
        f"{sample_name}: "
        f"row={row_index}, "
        f"image shape={center_image.shape}, "
        f"steering={row['steering']:.6f}"
    )

### Quantify Steering Imbalance

Calculate how many samples lie near zero steering and plot the complete steering distribution. This reveals whether a model could achieve deceptively low error by predicting straight driving most of the time.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

steering_values = data["steering"].to_numpy(dtype=np.float32)

thresholds = [0.01, 0.05, 0.10, 0.20, 0.40]

distribution_rows = []

for threshold in thresholds:
    count = np.sum(np.abs(steering_values) <= threshold)
    percentage = 100.0 * count / len(steering_values)

    distribution_rows.append(
        {
            "Absolute steering threshold": threshold,
            "Number of samples": int(count),
            "Percentage of records": percentage
        }
    )

distribution_summary = pd.DataFrame(distribution_rows)

print("Near-zero steering concentration:")
display(distribution_summary)

print("\nAdditional statistics:")
print(f"Median steering: {np.median(steering_values):.6f}")
print(
    f"Median absolute steering: "
    f"{np.median(np.abs(steering_values)):.6f}"
)
print(
    f"Standard deviation: "
    f"{np.std(steering_values):.6f}"
)

print(
    f"Left-turn records: "
    f"{np.sum(steering_values < -0.05)}"
)

print(
    f"Approximately straight records: "
    f"{np.sum(np.abs(steering_values) <= 0.05)}"
)

print(
    f"Right-turn records: "
    f"{np.sum(steering_values > 0.05)}"
)

plt.figure(figsize=(12, 5))

plt.hist(
    steering_values,
    bins=51,
    edgecolor="black"
)

plt.axvline(
    0,
    linestyle="--",
    linewidth=1.5,
    label="Zero steering"
)

plt.xlabel("Steering angle")
plt.ylabel("Number of driving records")
plt.title("Original Steering-Angle Distribution")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

### Define Steering Regions

Convert the continuous steering target into five analysis-only regions: sharp left, left, straight, right, and sharp right. These categories are not used as training labels; they support interpretable error analysis later.


In [ ]:
def assign_steering_region(steering):
    """
    Assign a readable driving category based on steering value.
    These labels are for analysis only, not model training.
    """

    if steering < -0.30:
        return "Sharp left"

    elif steering < -0.05:
        return "Left"

    elif steering <= 0.05:
        return "Straight"

    elif steering <= 0.30:
        return "Right"

    else:
        return "Sharp right"


data["steering_region"] = data["steering"].apply(
    assign_steering_region
)

region_order = [
    "Sharp left",
    "Left",
    "Straight",
    "Right",
    "Sharp right"
]

region_counts = (
    data["steering_region"]
    .value_counts()
    .reindex(region_order, fill_value=0)
)

region_percentages = (
    100 * region_counts / len(data)
)

region_summary = pd.DataFrame(
    {
        "Records": region_counts,
        "Percentage": region_percentages
    }
)

display(region_summary)

plt.figure(figsize=(10, 5))

plt.bar(
    region_summary.index,
    region_summary["Records"]
)

plt.xlabel("Steering region")
plt.ylabel("Number of driving records")
plt.title("Driving Records by Steering Region")
plt.xticks(rotation=20)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

### Create Leakage-Resistant Data Splits

Assign consecutive records to sequence groups and split entire groups into training, validation, and test subsets. Keeping nearby frames and synchronized camera views together reduces leakage from highly correlated simulator images.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
GROUP_SIZE = 25
TEST_FRACTION = 0.15
VALIDATION_FRACTION_OF_REMAINING = 0.1765

# 0.1765 of the remaining 85% is approximately 15% of all data.
# Expected final split:
# training   ≈ 70%
# validation ≈ 15%
# testing    ≈ 15%

split_data = data.copy().reset_index(drop=True)

# Consecutive records are placed into the same group.
split_data["sequence_group"] = (
    np.arange(len(split_data)) // GROUP_SIZE
)

print("Number of sequential groups:", split_data["sequence_group"].nunique())
print("Approximate records per group:", GROUP_SIZE)

# First split: train+validation versus test

test_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_FRACTION,
    random_state=SEED
)

train_val_indices, test_indices = next(
    test_splitter.split(
        split_data,
        groups=split_data["sequence_group"]
    )
)

train_val_data = (
    split_data
    .iloc[train_val_indices]
    .copy()
    .reset_index(drop=True)
)

test_data = (
    split_data
    .iloc[test_indices]
    .copy()
    .reset_index(drop=True)
)

# Second split: training versus validation

validation_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=VALIDATION_FRACTION_OF_REMAINING,
    random_state=SEED
)

train_indices, validation_indices = next(
    validation_splitter.split(
        train_val_data,
        groups=train_val_data["sequence_group"]
    )
)

train_data = (
    train_val_data
    .iloc[train_indices]
    .copy()
    .reset_index(drop=True)
)

validation_data = (
    train_val_data
    .iloc[validation_indices]
    .copy()
    .reset_index(drop=True)
)

# Verify that groups do not overlap

train_groups = set(train_data["sequence_group"])
validation_groups = set(validation_data["sequence_group"])
test_groups = set(test_data["sequence_group"])

assert train_groups.isdisjoint(validation_groups)
assert train_groups.isdisjoint(test_groups)
assert validation_groups.isdisjoint(test_groups)

split_summary = pd.DataFrame(
    {
        "Subset": ["Training", "Validation", "Test"],
        "Driving records": [
            len(train_data),
            len(validation_data),
            len(test_data)
        ],
        "Percentage": [
            100 * len(train_data) / len(split_data),
            100 * len(validation_data) / len(split_data),
            100 * len(test_data) / len(split_data)
        ],
        "Mean steering": [
            train_data["steering"].mean(),
            validation_data["steering"].mean(),
            test_data["steering"].mean()
        ],
        "Steering standard deviation": [
            train_data["steering"].std(),
            validation_data["steering"].std(),
            test_data["steering"].std()
        ]
    }
)

display(split_summary)

print("Training groups  :", len(train_groups))
print("Validation groups:", len(validation_groups))
print("Test groups      :", len(test_groups))

### Check Split Distributions

Compare normalized steering histograms and descriptive statistics across the three subsets. Similar coverage of left, straight, and right steering confirms that the grouped split did not accidentally remove important maneuvers from any subset.


In [ ]:
plt.figure(figsize=(12, 6))

plt.hist(
    train_data["steering"],
    bins=51,
    density=True,
    alpha=0.45,
    label="Training"
)

plt.hist(
    validation_data["steering"],
    bins=51,
    density=True,
    alpha=0.45,
    label="Validation"
)

plt.hist(
    test_data["steering"],
    bins=51,
    density=True,
    alpha=0.45,
    label="Test"
)

plt.xlabel("Steering angle")
plt.ylabel("Normalized density")
plt.title("Steering Distributions Across Data Splits")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

for subset_name, subset in [
    ("Training", train_data),
    ("Validation", validation_data),
    ("Test", test_data)
]:
    print(f"\n{subset_name}")
    print("-" * len(subset_name))
    print(f"Records: {len(subset)}")
    print(f"Minimum: {subset['steering'].min():.6f}")
    print(f"Maximum: {subset['steering'].max():.6f}")
    print(f"Mean   : {subset['steering'].mean():.6f}")
    print(f"Std.   : {subset['steering'].std():.6f}")

*   The dataset is extremely imbalanced towards straight driving
*   Right-left turn distribution is balanced
*   This will be dealt with through augmentation for training dataset





### Expand Training Records with Side Cameras

Create image-level training samples from the center, left, and right cameras. Side-camera labels receive steering corrections so they represent recovery behavior, while validation and test evaluation remain based on the normal center view.


In [ ]:
import numpy as np
import pandas as pd

# Steering correction for side cameras.
# We may tune this later, but 0.15 is a reasonable baseline.
CAMERA_CORRECTION = 0.15

def expand_camera_samples(
    dataframe,
    include_side_cameras=True,
    camera_correction=0.15
):
    """
    Convert driving records into image-path and steering-label samples.

    Center camera:
        steering remains unchanged.

    Left camera:
        positive correction is added because the vehicle should steer
        rightward to return toward the center of the lane.

    Right camera:
        negative correction is added because the vehicle should steer
        leftward to return toward the center of the lane.
    """

    samples = []

    for _, row in dataframe.iterrows():

        # Center camera
        samples.append(
            {
                "image_path": row["center_path"],
                "steering": float(row["steering"]),
                "camera": "center"
            }
        )

        if include_side_cameras:

            # Left camera
            samples.append(
                {
                    "image_path": row["left_path"],
                    "steering": float(
                        np.clip(
                            row["steering"] + camera_correction,
                            -1.0,
                            1.0
                        )
                    ),
                    "camera": "left"
                }
            )

            # Right camera
            samples.append(
                {
                    "image_path": row["right_path"],
                    "steering": float(
                        np.clip(
                            row["steering"] - camera_correction,
                            -1.0,
                            1.0
                        )
                    ),
                    "camera": "right"
                }
            )

    return pd.DataFrame(samples)


# Use all three cameras for training.
train_samples_full = expand_camera_samples(
    train_data,
    include_side_cameras=True,
    camera_correction=CAMERA_CORRECTION
)

# Use only the center camera for validation and testing.
# This measures performance on the actual forward-driving view.
validation_samples = expand_camera_samples(
    validation_data,
    include_side_cameras=False
)

test_samples = expand_camera_samples(
    test_data,
    include_side_cameras=False
)

print("Expanded sample counts:")
print("Training before balancing:", len(train_samples_full))
print("Validation:", len(validation_samples))
print("Test:", len(test_samples))

print("\nTraining camera counts:")
display(train_samples_full["camera"].value_counts())

print("\nExpanded training steering statistics:")
display(train_samples_full["steering"].describe())

### Balance the Training Set

Bin the expanded steering values and cap the number of samples retained in overrepresented bins. Balancing is applied only to training data so validation and test metrics continue to reflect the original driving distribution.


In [ ]:
import numpy as np
import pandas as pd

# Number of steering bins
NUM_BINS = 31

# Maximum samples retained in any one steering bin.
# This is intentionally not too aggressive because the dataset is modest.
MAX_SAMPLES_PER_BIN = 700

balanced_train_samples = train_samples_full.copy()

bin_counts, bin_edges = np.histogram(
    balanced_train_samples["steering"],
    bins=NUM_BINS,
    range=(-1.0, 1.0)
)

# Assign every sample to a bin.
balanced_train_samples["steering_bin"] = np.digitize(
    balanced_train_samples["steering"],
    bin_edges[1:-1],
    right=False
)

rng = np.random.default_rng(SEED)

indices_to_keep = []

for bin_id in range(NUM_BINS):

    bin_indices = balanced_train_samples.index[
        balanced_train_samples["steering_bin"] == bin_id
    ].to_numpy()

    if len(bin_indices) <= MAX_SAMPLES_PER_BIN:
        selected_indices = bin_indices
    else:
        selected_indices = rng.choice(
            bin_indices,
            size=MAX_SAMPLES_PER_BIN,
            replace=False
        )

    indices_to_keep.extend(selected_indices.tolist())

train_samples = (
    balanced_train_samples
    .loc[indices_to_keep]
    .sample(frac=1.0, random_state=SEED)
    .drop(columns="steering_bin")
    .reset_index(drop=True)
)

print("Training samples before balancing:", len(train_samples_full))
print("Training samples after balancing :", len(train_samples))
print(
    "Samples removed:",
    len(train_samples_full) - len(train_samples)
)

print("\nBalanced training steering statistics:")
display(train_samples["steering"].describe())

print("\nCamera counts after balancing:")
display(train_samples["camera"].value_counts())

### Visualize the Effect of Balancing

Plot steering histograms before and after undersampling and calculate the remaining near-zero percentage. This verifies that straight-driving dominance was reduced without eliminating normal straight-road behavior.


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.hist(
    train_samples_full["steering"],
    bins=51,
    range=(-1, 1),
    alpha=0.55,
    label="Before balancing"
)

plt.hist(
    train_samples["steering"],
    bins=51,
    range=(-1, 1),
    alpha=0.55,
    label="After balancing"
)

plt.xlabel("Steering angle")
plt.ylabel("Number of image samples")
plt.title("Training Steering Distribution Before and After Balancing")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

near_zero_before = np.mean(
    np.abs(train_samples_full["steering"]) <= 0.05
) * 100

near_zero_after = np.mean(
    np.abs(train_samples["steering"]) <= 0.05
) * 100

print(
    f"Near-zero percentage before balancing: "
    f"{near_zero_before:.2f}%"
)

print(
    f"Near-zero percentage after balancing : "
    f"{near_zero_after:.2f}%"
)

### Define NVIDIA-Style Preprocessing

Crop sky and vehicle hood, resize each image to 200 by 66, convert RGB to YUV, and normalize pixel values to approximately minus one to one. This matches the preprocessing used for the PilotNet-family models.


In [ ]:
import cv2
import numpy as np

INPUT_HEIGHT = 66
INPUT_WIDTH = 200
INPUT_CHANNELS = 3

# Crop settings for the original 160 x 320 images.
# Remove much of the sky and a small part of the vehicle hood.
CROP_TOP = 60
CROP_BOTTOM = 25

def load_image_rgb(image_path):
    """
    Load an image from disk and return it in RGB format.
    """

    image = cv2.imread(str(image_path))

    if image is None:
        raise FileNotFoundError(
            f"Unable to read image: {image_path}"
        )

    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


def preprocess_image(image):
    """
    Apply NVIDIA-style preprocessing:

    1. crop sky and hood,
    2. resize to 200 x 66,
    3. convert RGB to YUV,
    4. convert to float32,
    5. normalize to approximately [-1, 1].
    """

    height = image.shape[0]

    cropped = image[
        CROP_TOP : height - CROP_BOTTOM,
        :,
        :
    ]

    resized = cv2.resize(
        cropped,
        (INPUT_WIDTH, INPUT_HEIGHT),
        interpolation=cv2.INTER_AREA
    )

    yuv = cv2.cvtColor(
        resized,
        cv2.COLOR_RGB2YUV
    )

    normalized = (
        yuv.astype(np.float32) / 127.5
    ) - 1.0

    return normalized

### Verify the Preprocessing Pipeline

Display the original, cropped, resized, and processed image representations. The numerical checks confirm the final tensor shape, data type, and normalization range expected by the neural network.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sample_row = train_samples.iloc[
    np.random.default_rng(SEED).integers(
        0,
        len(train_samples)
    )
]

original_image = load_image_rgb(
    sample_row["image_path"]
)

cropped_image = original_image[
    CROP_TOP : original_image.shape[0] - CROP_BOTTOM
]

resized_rgb = cv2.resize(
    cropped_image,
    (INPUT_WIDTH, INPUT_HEIGHT),
    interpolation=cv2.INTER_AREA
)

processed_yuv = preprocess_image(original_image)

# Convert normalized YUV back to RGB for visualization only.
display_yuv = np.clip(
    (processed_yuv + 1.0) * 127.5,
    0,
    255
).astype(np.uint8)

display_rgb = cv2.cvtColor(
    display_yuv,
    cv2.COLOR_YUV2RGB
)

fig, axes = plt.subplots(1, 4, figsize=(17, 4))

axes[0].imshow(original_image)
axes[0].set_title(
    f"Original\n{original_image.shape}"
)
axes[0].axis("off")

axes[1].imshow(cropped_image)
axes[1].set_title(
    f"Cropped\n{cropped_image.shape}"
)
axes[1].axis("off")

axes[2].imshow(resized_rgb)
axes[2].set_title(
    f"Resized RGB\n{resized_rgb.shape}"
)
axes[2].axis("off")

axes[3].imshow(display_rgb)
axes[3].set_title(
    f"Processed YUV\n{processed_yuv.shape}"
)
axes[3].axis("off")

plt.tight_layout()
plt.show()

print("Sample camera:", sample_row["camera"])
print("Sample steering:", sample_row["steering"])
print("Processed shape:", processed_yuv.shape)
print("Processed dtype:", processed_yuv.dtype)
print("Minimum value:", processed_yuv.min())
print("Maximum value:", processed_yuv.max())

### Define Label-Consistent Augmentation

Implement random brightness changes, shadows, translation, and horizontal flipping. Steering labels are adjusted when translation or flipping changes the implied vehicle position, preventing inconsistent image-label pairs.


In [ ]:
import cv2
import numpy as np

# Horizontal pixel shift and its steering correction.
MAX_HORIZONTAL_SHIFT = 40
STEERING_PER_PIXEL = 0.002

def random_brightness(image, rng):
    """
    Randomly adjust image brightness in HSV color space.
    """

    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    hsv = hsv.astype(np.float32)

    brightness_factor = rng.uniform(0.6, 1.4)

    hsv[:, :, 2] *= brightness_factor
    hsv[:, :, 2] = np.clip(hsv[:, :, 2], 0, 255)

    hsv = hsv.astype(np.uint8)

    return cv2.cvtColor(
        hsv,
        cv2.COLOR_HSV2RGB
    )


def random_shadow(image, rng):
    """
    Add a simple random shadow across one side of the image.
    """

    height, width = image.shape[:2]

    x1 = rng.integers(0, width)
    x2 = rng.integers(0, width)

    yy, xx = np.mgrid[0:height, 0:width]

    mask = (
        (xx - x1)
        * (height - 1)
        - (x2 - x1)
        * yy
    ) > 0

    if rng.random() < 0.5:
        mask = ~mask

    shadowed = image.astype(np.float32)

    shadow_strength = rng.uniform(0.45, 0.75)
    shadowed[mask] *= shadow_strength

    return np.clip(
        shadowed,
        0,
        255
    ).astype(np.uint8)


def translate_image_and_steering(image, steering, rng):
    """
    Translate the image horizontally and adjust the steering label.
    """

    horizontal_shift = rng.uniform(
        -MAX_HORIZONTAL_SHIFT,
        MAX_HORIZONTAL_SHIFT
    )

    vertical_shift = rng.uniform(-5, 5)

    transformation_matrix = np.float32(
        [
            [1, 0, horizontal_shift],
            [0, 1, vertical_shift]
        ]
    )

    translated_image = cv2.warpAffine(
        image,
        transformation_matrix,
        (image.shape[1], image.shape[0]),
        borderMode=cv2.BORDER_REPLICATE
    )

    translated_steering = (
        steering
        + horizontal_shift * STEERING_PER_PIXEL
    )

    translated_steering = float(
        np.clip(
            translated_steering,
            -1.0,
            1.0
        )
    )

    return translated_image, translated_steering


def augment_image(image, steering, rng):
    """
    Apply random training-only augmentation.
    """

    augmented_image = image.copy()
    augmented_steering = float(steering)

    # Horizontal translation
    if rng.random() < 0.5:
        augmented_image, augmented_steering = (
            translate_image_and_steering(
                augmented_image,
                augmented_steering,
                rng
            )
        )

    # Brightness variation
    if rng.random() < 0.5:
        augmented_image = random_brightness(
            augmented_image,
            rng
        )

    # Random shadow
    if rng.random() < 0.3:
        augmented_image = random_shadow(
            augmented_image,
            rng
        )

    # Horizontal flip
    if rng.random() < 0.5:
        augmented_image = cv2.flip(
            augmented_image,
            1
        )

        augmented_steering = -augmented_steering

    return augmented_image, augmented_steering

### Inspect Augmentation Examples

Generate multiple random transformations from one training image and display their updated steering values. This confirms that augmentations remain visually plausible and that geometric transformations modify labels correctly.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sample = train_samples.sample(
    n=1,
    random_state=SEED
).iloc[0]

base_image = load_image_rgb(
    sample["image_path"]
)

base_steering = float(
    sample["steering"]
)

fig, axes = plt.subplots(
    2,
    4,
    figsize=(16, 8)
)

axes = axes.flatten()

axes[0].imshow(base_image)
axes[0].set_title(
    f"Original\nsteering={base_steering:.3f}"
)
axes[0].axis("off")

for index in range(1, 8):

    rng = np.random.default_rng(
        SEED + index
    )

    augmented_image, augmented_steering = (
        augment_image(
            base_image,
            base_steering,
            rng
        )
    )

    axes[index].imshow(augmented_image)
    axes[index].set_title(
        f"Augmentation {index}\n"
        f"steering={augmented_steering:.3f}"
    )
    axes[index].axis("off")

plt.tight_layout()
plt.show()

### Build TensorFlow Input Pipelines

Wrap OpenCV loading and augmentation inside `tf.data` pipelines. Training samples are shuffled and augmented, evaluation samples are processed deterministically, and batching plus prefetching improves throughput.


In [ ]:
import tensorflow as tf
import numpy as np

BATCH_SIZE = 64

AUTOTUNE = tf.data.AUTOTUNE

# NumPy-side image loading functions

def load_and_process_training_sample(path_bytes, steering_value):
    """
    Load, augment, and preprocess one training sample.

    This function runs through tf.numpy_function because it uses
    OpenCV and NumPy-based augmentation.
    """

    image_path = path_bytes.decode("utf-8")
    steering = float(steering_value)

    image = load_image_rgb(image_path)

    # A new random generator is created for each call.
    rng = np.random.default_rng()

    image, steering = augment_image(
        image,
        steering,
        rng
    )

    image = preprocess_image(image)

    return (
        image.astype(np.float32),
        np.float32(steering)
    )


def load_and_process_evaluation_sample(path_bytes, steering_value):
    """
    Load and preprocess one validation or test sample.

    No augmentation is applied.
    """

    image_path = path_bytes.decode("utf-8")
    steering = float(steering_value)

    image = load_image_rgb(image_path)
    image = preprocess_image(image)

    return (
        image.astype(np.float32),
        np.float32(steering)
    )

# TensorFlow wrappers

def tf_process_training_sample(image_path, steering):
    image, steering = tf.numpy_function(
        func=load_and_process_training_sample,
        inp=[image_path, steering],
        Tout=[tf.float32, tf.float32]
    )

    image.set_shape(
        [INPUT_HEIGHT, INPUT_WIDTH, INPUT_CHANNELS]
    )

    steering.set_shape([])

    return image, steering


def tf_process_evaluation_sample(image_path, steering):
    image, steering = tf.numpy_function(
        func=load_and_process_evaluation_sample,
        inp=[image_path, steering],
        Tout=[tf.float32, tf.float32]
    )

    image.set_shape(
        [INPUT_HEIGHT, INPUT_WIDTH, INPUT_CHANNELS]
    )

    steering.set_shape([])

    return image, steering

# Dataset-construction function

def create_dataset(
    samples_dataframe,
    training=False,
    batch_size=64
):
    image_paths = samples_dataframe[
        "image_path"
    ].astype(str).to_numpy()

    steering_values = samples_dataframe[
        "steering"
    ].astype(np.float32).to_numpy()

    dataset = tf.data.Dataset.from_tensor_slices(
        (image_paths, steering_values)
    )

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(samples_dataframe),
            seed=SEED,
            reshuffle_each_iteration=True
        )

        dataset = dataset.map(
            tf_process_training_sample,
            num_parallel_calls=AUTOTUNE
        )

    else:
        dataset = dataset.map(
            tf_process_evaluation_sample,
            num_parallel_calls=AUTOTUNE
        )

    dataset = dataset.batch(
        batch_size,
        drop_remainder=False
    )

    dataset = dataset.prefetch(AUTOTUNE)

    return dataset


train_dataset = create_dataset(
    train_samples,
    training=True,
    batch_size=BATCH_SIZE
)

validation_dataset = create_dataset(
    validation_samples,
    training=False,
    batch_size=BATCH_SIZE
)

test_dataset = create_dataset(
    test_samples,
    training=False,
    batch_size=BATCH_SIZE
)

print("TensorFlow datasets created successfully.")

print("\nTraining batches:")
print(tf.data.experimental.cardinality(train_dataset).numpy())

print("Validation batches:")
print(tf.data.experimental.cardinality(validation_dataset).numpy())

print("Test batches:")
print(tf.data.experimental.cardinality(test_dataset).numpy())

### Verify a Training Batch

Retrieve one batch and inspect tensor shapes, data types, value ranges, and sample images. This catches pipeline problems before training begins and confirms the model will receive tensors of shape 66 by 200 by 3.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sample_images, sample_steerings = next(
    iter(train_dataset)
)

print("Image batch shape:", sample_images.shape)
print("Steering batch shape:", sample_steerings.shape)
print("Image dtype:", sample_images.dtype)
print("Steering dtype:", sample_steerings.dtype)

print(
    "Image value range:",
    float(tf.reduce_min(sample_images)),
    "to",
    float(tf.reduce_max(sample_images))
)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for index, axis in enumerate(axes.flatten()):

    image_yuv = sample_images[index].numpy()

    # Undo normalization for visualization.
    image_yuv = np.clip(
        (image_yuv + 1.0) * 127.5,
        0,
        255
    ).astype(np.uint8)

    image_rgb = cv2.cvtColor(
        image_yuv,
        cv2.COLOR_YUV2RGB
    )

    axis.imshow(image_rgb)

    axis.set_title(
        f"Steering = "
        f"{sample_steerings[index].numpy():.3f}"
    )

    axis.axis("off")

plt.tight_layout()
plt.show()

### Define the PilotNet Baseline

Construct the NVIDIA-inspired convolutional regression network with five convolutional layers, ELU activations, dense layers, dropout, and a linear steering output. The model is compiled with Adam, mean-squared error loss, MAE, and RMSE.


In [ ]:
from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    Flatten,
    Dense,
    Dropout
)
from tensorflow.keras.optimizers import Adam

def build_pilotnet_model(
    learning_rate=1e-4,
    dropout_rate=0.5
):
    """
    Construct a PilotNet-style steering-angle regression model.
    """

    inputs = Input(
        shape=(
            INPUT_HEIGHT,
            INPUT_WIDTH,
            INPUT_CHANNELS
        ),
        name="input_image"
    )

    # Five convolutional layers
    x = Conv2D(
        filters=24,
        kernel_size=(5, 5),
        strides=(2, 2),
        activation="elu",
        name="conv_1"
    )(inputs)

    x = Conv2D(
        filters=36,
        kernel_size=(5, 5),
        strides=(2, 2),
        activation="elu",
        name="conv_2"
    )(x)

    x = Conv2D(
        filters=48,
        kernel_size=(5, 5),
        strides=(2, 2),
        activation="elu",
        name="conv_3"
    )(x)

    x = Conv2D(
        filters=64,
        kernel_size=(3, 3),
        strides=(1, 1),
        activation="elu",
        name="conv_4"
    )(x)

    x = Conv2D(
        filters=64,
        kernel_size=(3, 3),
        strides=(1, 1),
        activation="elu",
        name="conv_5"
    )(x)

    x = Flatten(
        name="flatten"
    )(x)

    x = Dense(
        100,
        activation="elu",
        name="dense_100"
    )(x)

    x = Dropout(
        dropout_rate,
        name="dropout_1"
    )(x)

    x = Dense(
        50,
        activation="elu",
        name="dense_50"
    )(x)

    x = Dropout(
        dropout_rate * 0.5,
        name="dropout_2"
    )(x)

    x = Dense(
        10,
        activation="elu",
        name="dense_10"
    )(x)

    outputs = Dense(
        1,
        activation="linear",
        name="steering_output"
    )(x)

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name="PilotNet_Baseline"
    )

    model.compile(
        optimizer=Adam(
            learning_rate=learning_rate
        ),
        loss="mse",
        metrics=[
            tf.keras.metrics.MeanAbsoluteError(
                name="mae"
            ),
            tf.keras.metrics.RootMeanSquaredError(
                name="rmse"
            )
        ]
    )

    return model


pilotnet_model = build_pilotnet_model(
    learning_rate=1e-4,
    dropout_rate=0.5
)

pilotnet_model.summary()

### Configure Baseline Training Controls

Create callbacks for checkpointing, early stopping, learning-rate reduction, CSV logging, and NaN termination. These callbacks preserve the best validation checkpoint and prevent unnecessary epochs after improvement stops.


In [ ]:
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint,
    CSVLogger,
    TerminateOnNaN
)

MODEL_DIR = PROJECT_DIR / "trained_models"
RESULTS_DIR = PROJECT_DIR / "results"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PILOTNET_MODEL_PATH = (
    MODEL_DIR / "pilotnet_baseline_best.keras"
)

PILOTNET_LOG_PATH = (
    RESULTS_DIR / "pilotnet_baseline_training.csv"
)

pilotnet_callbacks = [
    ModelCheckpoint(
        filepath=str(PILOTNET_MODEL_PATH),
        monitor="val_mae",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    EarlyStopping(
        monitor="val_mae",
        mode="min",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_mae",
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),

    CSVLogger(
        filename=str(PILOTNET_LOG_PATH),
        append=False
    ),

    TerminateOnNaN()
]

print("Best model will be saved to:")
print(PILOTNET_MODEL_PATH)

print("\nTraining log will be saved to:")
print(PILOTNET_LOG_PATH)

### Train the PilotNet Baseline

Fit the baseline on augmented training batches and evaluate after each epoch on clean center-camera validation data. The elapsed training time is recorded for later efficiency comparisons.


In [ ]:
import time

MAX_EPOCHS = 25

training_start_time = time.perf_counter()

pilotnet_history = pilotnet_model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=MAX_EPOCHS,
    callbacks=pilotnet_callbacks,
    verbose=1
)

pilotnet_training_time = (
    time.perf_counter() - training_start_time
)

print(
    f"\nTotal PilotNet training time: "
    f"{pilotnet_training_time / 60:.2f} minutes"
)

### Analyze PilotNet Learning Curves

Plot training and validation MSE and MAE over epochs, then identify the best validation epoch. Differences between training and validation curves should be interpreted in light of augmentation and dropout being active only during training.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

history_data = pilotnet_history.history

epochs_completed = range(
    1,
    len(history_data["loss"]) + 1
)

plt.figure(figsize=(10, 5))

plt.plot(
    epochs_completed,
    history_data["loss"],
    marker="o",
    label="Training MSE"
)

plt.plot(
    epochs_completed,
    history_data["val_loss"],
    marker="o",
    label="Validation MSE"
)

plt.xlabel("Epoch")
plt.ylabel("Mean squared error")
plt.title("PilotNet Training and Validation Loss")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


plt.figure(figsize=(10, 5))

plt.plot(
    epochs_completed,
    history_data["mae"],
    marker="o",
    label="Training MAE"
)

plt.plot(
    epochs_completed,
    history_data["val_mae"],
    marker="o",
    label="Validation MAE"
)

plt.xlabel("Epoch")
plt.ylabel("Mean absolute error")
plt.title("PilotNet Training and Validation MAE")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


best_epoch = (
    int(np.argmin(history_data["val_mae"])) + 1
)

print("Best epoch based on validation MAE:", best_epoch)

print(
    "Best validation MAE:",
    min(history_data["val_mae"])
)

print(
    "Validation RMSE at best epoch:",
    history_data["val_rmse"][best_epoch - 1]
)

### Interpretation of the Baseline
The validation loss is lower than the training loss, this is so because the training dataset has more variations due to the augmentations. We have implemented different augmentation techniques including:

1.   Random translations
2.   Brightness changes
3.   Shadows
4.   Flips (Horizontal)
5.   Side camera images
6.   Drop-out during training

Validation uses clean center-camera images with dropout disabled. Therefore, training and validation losses are not measured under equally difficult conditions.

Also, the validation performance did not improve after the first epoch. The possible reasons for this might be:

1.   The model learned the steering relationships very early
2.   The training distribution is substantially different from validation because training uses side cameras while validation uses only the center camera.
3.   Learning rate drop out may be suboptimal
4.   The 5,260-sample balanced training set may be smaller than necessary.


### Reload and Evaluate the Best Baseline

Load the checkpoint selected by validation MAE and evaluate it on validation and untouched test datasets. Reloading ensures that final metrics come from the best saved epoch rather than the last epoch.


In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import time

# Load the checkpoint selected using validation MAE.
best_pilotnet_model = tf.keras.models.load_model(
    str(PILOTNET_MODEL_PATH)
)

print("Best PilotNet checkpoint loaded successfully.")
print("Model path:", PILOTNET_MODEL_PATH)

validation_results = best_pilotnet_model.evaluate(
    validation_dataset,
    verbose=1,
    return_dict=True
)

test_start_time = time.perf_counter()

test_results = best_pilotnet_model.evaluate(
    test_dataset,
    verbose=1,
    return_dict=True
)

test_evaluation_time = time.perf_counter() - test_start_time

evaluation_table = pd.DataFrame(
    [
        {
            "Model": "PilotNet baseline",
            "Subset": "Validation",
            "MSE": validation_results["loss"],
            "MAE": validation_results["mae"],
            "RMSE": validation_results["rmse"]
        },
        {
            "Model": "PilotNet baseline",
            "Subset": "Test",
            "MSE": test_results["loss"],
            "MAE": test_results["mae"],
            "RMSE": test_results["rmse"]
        }
    ]
)

display(evaluation_table)

print(
    f"Test-set evaluation time: "
    f"{test_evaluation_time:.2f} seconds"
)

### Compute Complete Baseline Test Metrics

Generate test predictions and calculate MAE, MSE, RMSE, R-squared, and prediction bias. The results are stored in a table and written to disk for later model comparison.


In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# The test dataset was not shuffled, so its prediction order
# matches test_samples.
test_predictions = best_pilotnet_model.predict(
    test_dataset,
    verbose=1
).reshape(-1)

test_actual = (
    test_samples["steering"]
    .to_numpy(dtype=np.float32)
)

assert len(test_predictions) == len(test_actual)

test_mae = mean_absolute_error(
    test_actual,
    test_predictions
)

test_mse = mean_squared_error(
    test_actual,
    test_predictions
)

test_rmse = np.sqrt(test_mse)

test_r2 = r2_score(
    test_actual,
    test_predictions
)

test_bias = np.mean(
    test_predictions - test_actual
)

test_metrics = pd.DataFrame(
    [
        {
            "Model": "PilotNet baseline",
            "Parameters": best_pilotnet_model.count_params(),
            "Training time (minutes)": (
                pilotnet_training_time / 60
            ),
            "Test MAE": test_mae,
            "Test MSE": test_mse,
            "Test RMSE": test_rmse,
            "Test R2": test_r2,
            "Prediction bias": test_bias
        }
    ]
)

display(test_metrics)

# Save results for the final model comparison.
PILOTNET_METRICS_PATH = (
    RESULTS_DIR / "pilotnet_baseline_metrics.csv"
)

test_metrics.to_csv(
    PILOTNET_METRICS_PATH,
    index=False
)

print("Metrics saved to:")
print(PILOTNET_METRICS_PATH)

### Visualize Baseline Predictions

Plot actual versus predicted steering, residuals, and a time-ordered segment of the test sequence. These plots reveal regression toward zero, systematic bias, and whether the model follows rapid steering changes.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

residuals = test_actual - test_predictions

# ------------------------------------------------------------
# Plot 1: actual versus predicted
# ------------------------------------------------------------
plt.figure(figsize=(7, 7))

plt.scatter(
    test_actual,
    test_predictions,
    alpha=0.4,
    s=18
)

minimum_value = min(
    test_actual.min(),
    test_predictions.min()
)

maximum_value = max(
    test_actual.max(),
    test_predictions.max()
)

plt.plot(
    [minimum_value, maximum_value],
    [minimum_value, maximum_value],
    linestyle="--",
    label="Ideal prediction"
)

plt.xlabel("Actual steering angle")
plt.ylabel("Predicted steering angle")
plt.title("PilotNet: Actual versus Predicted Steering")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Plot 2: residual distribution
# ------------------------------------------------------------
plt.figure(figsize=(10, 5))

plt.hist(
    residuals,
    bins=40,
    edgecolor="black"
)

plt.axvline(
    0,
    linestyle="--",
    label="Zero residual"
)

plt.xlabel("Residual: actual − predicted")
plt.ylabel("Number of samples")
plt.title("PilotNet Test Residual Distribution")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Plot 3: predictions across the test sequence
# ------------------------------------------------------------
number_to_plot = min(
    300,
    len(test_actual)
)

plt.figure(figsize=(14, 5))

plt.plot(
    test_actual[:number_to_plot],
    label="Actual steering",
    linewidth=1.5
)

plt.plot(
    test_predictions[:number_to_plot],
    label="Predicted steering",
    linewidth=1.5
)

plt.xlabel("Test sample")
plt.ylabel("Steering angle")
plt.title(
    f"PilotNet Steering Predictions for First "
    f"{number_to_plot} Test Samples"
)

plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### Evaluate Baseline Performance by Steering Region

Calculate separate MAE and RMSE values for sharp-left, left, straight, right, and sharp-right samples. Region-level metrics expose rare but safety-critical failures hidden by an overall average.


In [ ]:
prediction_analysis = pd.DataFrame(
    {
        "actual": test_actual,
        "predicted": test_predictions
    }
)

prediction_analysis["absolute_error"] = np.abs(
    prediction_analysis["actual"]
    - prediction_analysis["predicted"]
)

prediction_analysis["steering_region"] = (
    prediction_analysis["actual"].apply(
        assign_steering_region
    )
)

region_performance_rows = []

for region in region_order:

    region_data = prediction_analysis[
        prediction_analysis["steering_region"] == region
    ]

    if len(region_data) == 0:
        continue

    region_rmse = np.sqrt(
        mean_squared_error(
            region_data["actual"],
            region_data["predicted"]
        )
    )

    region_performance_rows.append(
        {
            "Steering region": region,
            "Test samples": len(region_data),
            "MAE": region_data[
                "absolute_error"
            ].mean(),
            "RMSE": region_rmse,
            "Mean actual": region_data[
                "actual"
            ].mean(),
            "Mean predicted": region_data[
                "predicted"
            ].mean()
        }
    )

region_performance = pd.DataFrame(
    region_performance_rows
)

display(region_performance)

plt.figure(figsize=(10, 5))

plt.bar(
    region_performance["Steering region"],
    region_performance["MAE"]
)

plt.xlabel("Steering region")
plt.ylabel("Mean absolute error")
plt.title("PilotNet Test MAE by Steering Region")
plt.xticks(rotation=20)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

### Copy the Dataset to Local Colab Storage

Transfer and extract the zipped dataset into `/content` to avoid repeated reads from mounted Google Drive. Reading thousands of small image files locally can reduce training time dramatically.


In [ ]:
from pathlib import Path
import shutil
import time
import os

DRIVE_ZIP_PATH = Path(
    "/content/drive/MyDrive/ML_Python_Project.zip"
)

LOCAL_ZIP_PATH = Path(
    "/content/ML_Python_Project.zip"
)

LOCAL_PROJECT_DIR = Path(
    "/content/ML_Python_Project"
)

if not DRIVE_ZIP_PATH.exists():
    raise FileNotFoundError(
        f"ZIP archive not found:\n{DRIVE_ZIP_PATH}"
    )

start_time = time.perf_counter()

# Remove an incomplete local extraction left by the cancelled copy.
if LOCAL_PROJECT_DIR.exists():
    print("Removing incomplete local dataset...")
    shutil.rmtree(LOCAL_PROJECT_DIR)

if LOCAL_ZIP_PATH.exists():
    LOCAL_ZIP_PATH.unlink()

print("Copying the single ZIP archive from Drive...")

shutil.copy2(
    DRIVE_ZIP_PATH,
    LOCAL_ZIP_PATH
)

print("Extracting the archive locally...")

shutil.unpack_archive(
    str(LOCAL_ZIP_PATH),
    "/content"
)

elapsed_time = time.perf_counter() - start_time

print("\nTransfer and extraction completed.")
print(f"Total time: {elapsed_time:.2f} seconds")
print("Expected local folder:", LOCAL_PROJECT_DIR)
print("Folder exists:", LOCAL_PROJECT_DIR.exists())

### Inspect the Extracted Folder Structure

List the extracted directories to determine whether the archive created the expected project folder or an additional nested directory.


In [ ]:
# Inspect extracted folders
!find /content/ML_Python_Project -maxdepth 2 -type d | head -20

### Locate the Local Driving Log

Search the local Colab filesystem for `driving_log.csv` and select the copy outside Google Drive. This makes the local dataset path independent of the archive's internal folder structure.


In [ ]:
from pathlib import Path

CONTENT_DIR = Path("/content")

# Search only outside the mounted Google Drive directory.
local_driving_logs = [
    path for path in CONTENT_DIR.rglob("driving_log.csv")
    if "/content/drive/" not in str(path)
]

print("Local driving_log.csv files found:", len(local_driving_logs))

for index, csv_path in enumerate(local_driving_logs):
    print(f"[{index}] {csv_path}")

if len(local_driving_logs) == 0:
    print("\nTop-level contents of /content:")

    for item in sorted(CONTENT_DIR.iterdir()):
        print(
            "Folder:" if item.is_dir() else "File:  ",
            item
        )

    raise FileNotFoundError(
        "The archive was extracted, but driving_log.csv "
        "could not be found."
    )

# Use the first local driving log.
LOCAL_CSV_PATH = local_driving_logs[0]
LOCAL_PROJECT_DIR = LOCAL_CSV_PATH.parent

print("\nDetected local project directory:")
print(LOCAL_PROJECT_DIR)

print("\nDetected local CSV:")
print(LOCAL_CSV_PATH)

### Verify Local Images

Count locally extracted image files, display examples, and confirm that the expected dataset contents are present before remapping any paths.


In [ ]:
image_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp"
}

local_image_files = [
    path
    for path in LOCAL_PROJECT_DIR.rglob("*")
    if path.is_file()
    and path.suffix.lower() in image_extensions
]

print("Local project directory:", LOCAL_PROJECT_DIR)
print("Local images found:", len(local_image_files))

print("\nFirst five local images:")

for image_path in local_image_files[:5]:
    print(image_path)

print("\nContents of detected project directory:")

for item in sorted(LOCAL_PROJECT_DIR.iterdir()):
    item_type = "Folder" if item.is_dir() else "File"
    print(f"{item_type:7s}: {item.name}")

if len(local_image_files) < 21321:
    raise ValueError(
        f"Only {len(local_image_files)} images were found. "
        "Approximately 21,325 were expected."
    )

print("\nLocal dataset verification passed.")

### Remap DataFrames to Local Image Paths

Create a filename lookup for the extracted images and replace Drive-based paths in every sample DataFrame. Assertions ensure that no training, validation, or test image became unresolved.


In [ ]:
from pathlib import Path

local_image_lookup = {}

for image_path in local_image_files:
    filename = image_path.name

    if filename not in local_image_lookup:
        local_image_lookup[filename] = str(image_path)

print(
    "Unique image filenames indexed:",
    len(local_image_lookup)
)


def remap_to_local_path(existing_path):
    """
    Extract an image filename from its old Drive path and
    return the corresponding local Colab path.
    """

    filename = Path(
        str(existing_path).replace("\\", "/")
    ).name

    return local_image_lookup.get(filename)


sample_dataframes = {
    "Balanced training": train_samples,
    "Full training": train_samples_full,
    "Validation": validation_samples,
    "Test": test_samples
}

for name, dataframe in sample_dataframes.items():

    dataframe["image_path"] = (
        dataframe["image_path"]
        .apply(remap_to_local_path)
    )

    missing_count = dataframe["image_path"].isna().sum()

    print(
        f"{name:20s}: "
        f"{missing_count} missing paths"
    )

    if missing_count > 0:
        raise FileNotFoundError(
            f"Failed to remap {missing_count} paths "
            f"in {name}."
        )

print("\nExample remapped training path:")
print(train_samples.loc[0, "image_path"])

print("\nAll paths remapped successfully.")

### Rebuild Datasets with Local Paths

Recreate the TensorFlow training, validation, and test pipelines after path remapping so subsequent model training reads images from fast local storage.


In [ ]:
train_dataset = create_dataset(
    train_samples,
    training=True,
    batch_size=BATCH_SIZE
)

validation_dataset = create_dataset(
    validation_samples,
    training=False,
    batch_size=BATCH_SIZE
)

test_dataset = create_dataset(
    test_samples,
    training=False,
    batch_size=BATCH_SIZE
)

### Benchmark Local Data Loading

Iterate through a fixed number of training batches and report the average loading time per batch. This quantifies the improvement from moving the image files out of Google Drive.


In [ ]:
import time

train_dataset = create_dataset(
    train_samples,
    training=True,
    batch_size=BATCH_SIZE
)

validation_dataset = create_dataset(
    validation_samples,
    training=False,
    batch_size=BATCH_SIZE
)

test_dataset = create_dataset(
    test_samples,
    training=False,
    batch_size=BATCH_SIZE
)

batches_to_test = 20

benchmark_start = time.perf_counter()

for batch_number, _ in enumerate(train_dataset):
    if batch_number + 1 >= batches_to_test:
        break

benchmark_time = time.perf_counter() - benchmark_start

print(
    f"Time for {batches_to_test} training batches: "
    f"{benchmark_time:.2f} seconds"
)

print(
    f"Average time per batch: "
    f"{benchmark_time / batches_to_test:.3f} seconds"
)

### Create Steering-Dependent Sample Weights

Assign larger training weights to samples with larger absolute steering values. The aim is to make turning examples contribute more strongly to the objective without changing validation or test distributions.


In [ ]:
import numpy as np
import pandas as pd

weighted_train_samples = train_samples.copy()

weighted_train_samples["sample_weight"] = (
    1.0
    + 4.0 * np.abs(
        weighted_train_samples["steering"]
    )
)

weighted_train_samples["sample_weight"] = (
    weighted_train_samples["sample_weight"]
    .clip(upper=4.0)
)

weighted_train_samples["steering_region"] = (
    weighted_train_samples["steering"]
    .apply(assign_steering_region)
)

print("Sample-weight statistics:")
display(
    weighted_train_samples[
        ["steering", "sample_weight"]
    ].describe()
)

print("\nMean weight by steering region:")
display(
    weighted_train_samples
    .groupby("steering_region")["sample_weight"]
    .agg(["count", "mean", "min", "max"])
    .reindex(region_order)
)

### Build a Weighted TensorFlow Dataset

Extend the training pipeline to return `(image, target, sample_weight)` tuples. Keras automatically uses the third element to scale each sample's contribution to the loss.


In [ ]:
def create_weighted_training_dataset(
    samples_dataframe,
    batch_size=64
):
    image_paths = (
        samples_dataframe["image_path"]
        .astype(str)
        .to_numpy()
    )

    steering_values = (
        samples_dataframe["steering"]
        .astype(np.float32)
        .to_numpy()
    )

    sample_weights = (
        samples_dataframe["sample_weight"]
        .astype(np.float32)
        .to_numpy()
    )

    dataset = tf.data.Dataset.from_tensor_slices(
        (
            image_paths,
            steering_values,
            sample_weights
        )
    )

    dataset = dataset.shuffle(
        buffer_size=len(samples_dataframe),
        seed=SEED,
        reshuffle_each_iteration=True
    )

    def process_weighted_sample(
        image_path,
        steering,
        sample_weight
    ):
        image, processed_steering = (
            tf_process_training_sample(
                image_path,
                steering
            )
        )

        return (
            image,
            processed_steering,
            sample_weight
        )

    dataset = dataset.map(
        process_weighted_sample,
        num_parallel_calls=AUTOTUNE
    )

    dataset = dataset.batch(
        batch_size,
        drop_remainder=False
    )

    dataset = dataset.prefetch(AUTOTUNE)

    return dataset


weighted_train_dataset = (
    create_weighted_training_dataset(
        weighted_train_samples,
        batch_size=BATCH_SIZE
    )
)

images_batch, labels_batch, weights_batch = next(
    iter(weighted_train_dataset)
)

print("Image batch shape :", images_batch.shape)
print("Label batch shape :", labels_batch.shape)
print("Weight batch shape:", weights_batch.shape)

print(
    "Sample-weight range:",
    float(tf.reduce_min(weights_batch)),
    "to",
    float(tf.reduce_max(weights_batch))
)

### Define the Improved Regularized CNN

Build a compact alternative using convolution-batch-normalization-ELU blocks, spatial dropout, global average pooling, L2 regularization, and Huber loss. The architecture reduces parameter count while attempting to improve robustness and turn sensitivity.


In [ ]:
from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    BatchNormalization,
    ELU,
    SpatialDropout2D,
    GlobalAveragePooling2D,
    Dense,
    Dropout
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam

def convolution_block(
    inputs,
    filters,
    kernel_size,
    strides,
    l2_strength,
    spatial_dropout=0.0,
    block_name="block"
):
    x = Conv2D(
        filters=filters,
        kernel_size=kernel_size,
        strides=strides,
        padding="valid",
        use_bias=False,
        kernel_regularizer=l2(l2_strength),
        name=f"{block_name}_conv"
    )(inputs)

    x = BatchNormalization(
        name=f"{block_name}_batch_norm"
    )(x)

    x = ELU(
        name=f"{block_name}_elu"
    )(x)

    if spatial_dropout > 0:
        x = SpatialDropout2D(
            spatial_dropout,
            name=f"{block_name}_spatial_dropout"
        )(x)

    return x


def build_improved_cnn(
    learning_rate=3e-4,
    l2_strength=1e-5,
    dropout_rate=0.25
):
    inputs = Input(
        shape=(
            INPUT_HEIGHT,
            INPUT_WIDTH,
            INPUT_CHANNELS
        ),
        name="input_image"
    )

    x = convolution_block(
        inputs,
        filters=24,
        kernel_size=5,
        strides=2,
        l2_strength=l2_strength,
        block_name="block_1"
    )

    x = convolution_block(
        x,
        filters=36,
        kernel_size=5,
        strides=2,
        l2_strength=l2_strength,
        block_name="block_2"
    )

    x = convolution_block(
        x,
        filters=48,
        kernel_size=5,
        strides=2,
        l2_strength=l2_strength,
        spatial_dropout=0.05,
        block_name="block_3"
    )

    x = convolution_block(
        x,
        filters=64,
        kernel_size=3,
        strides=1,
        l2_strength=l2_strength,
        spatial_dropout=0.05,
        block_name="block_4"
    )

    x = convolution_block(
        x,
        filters=64,
        kernel_size=3,
        strides=1,
        l2_strength=l2_strength,
        block_name="block_5"
    )

    x = GlobalAveragePooling2D(
        name="global_average_pooling"
    )(x)

    x = Dense(
        64,
        activation="elu",
        kernel_regularizer=l2(l2_strength),
        name="dense_64"
    )(x)

    x = Dropout(
        dropout_rate,
        name="dropout"
    )(x)

    x = Dense(
        16,
        activation="elu",
        kernel_regularizer=l2(l2_strength),
        name="dense_16"
    )(x)

    outputs = Dense(
        1,
        activation="linear",
        name="steering_output"
    )(x)

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name="Improved_Weighted_CNN"
    )

    model.compile(
        optimizer=Adam(
            learning_rate=learning_rate
        ),
        loss=tf.keras.losses.Huber(
            delta=0.15
        ),
        metrics=[
            tf.keras.metrics.MeanAbsoluteError(
                name="mae"
            ),
            tf.keras.metrics.RootMeanSquaredError(
                name="rmse"
            )
        ]
    )

    return model


improved_cnn_model = build_improved_cnn()

improved_cnn_model.summary()

### Train the Improved CNN

Train the weighted model with checkpointing, early stopping, and learning-rate reduction. The best validation MAE, corresponding RMSE, epoch count, and elapsed time are recorded.


In [ ]:
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint,
    CSVLogger,
    TerminateOnNaN
)

import time

IMPROVED_MODEL_PATH = (
    MODEL_DIR / "improved_weighted_cnn_best.keras"
)

IMPROVED_LOG_PATH = (
    RESULTS_DIR / "improved_weighted_cnn_training.csv"
)

improved_callbacks = [
    ModelCheckpoint(
        filepath=str(IMPROVED_MODEL_PATH),
        monitor="val_mae",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    EarlyStopping(
        monitor="val_mae",
        mode="min",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_mae",
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),

    CSVLogger(
        filename=str(IMPROVED_LOG_PATH),
        append=False
    ),

    TerminateOnNaN()
]

training_start_time = time.perf_counter()

improved_history = improved_cnn_model.fit(
    weighted_train_dataset,
    validation_data=validation_dataset,
    epochs=25,
    callbacks=improved_callbacks,
    verbose=1
)

improved_training_time = (
    time.perf_counter() - training_start_time
)

best_improved_epoch = (
    int(
        np.argmin(
            improved_history.history["val_mae"]
        )
    ) + 1
)

print(
    f"\nImproved CNN training time: "
    f"{improved_training_time / 60:.2f} minutes"
)

print("Best epoch:", best_improved_epoch)

print(
    "Best validation MAE:",
    improved_history.history["val_mae"][
        best_improved_epoch - 1
    ]
)

print(
    "Validation RMSE at best epoch:",
    improved_history.history["val_rmse"][
        best_improved_epoch - 1
    ]
)

### Evaluate the Best Improved CNN

Reload the selected checkpoint and evaluate it on validation and test datasets. Huber loss is reported for this model, while MAE and RMSE remain directly comparable with other models.


In [ ]:
import tensorflow as tf
import pandas as pd
import time

best_improved_model = tf.keras.models.load_model(
    str(IMPROVED_MODEL_PATH)
)

print("Improved CNN checkpoint loaded successfully.")
print("Model parameters:", best_improved_model.count_params())

validation_results_improved = best_improved_model.evaluate(
    validation_dataset,
    verbose=1,
    return_dict=True
)

test_start_time = time.perf_counter()

test_results_improved = best_improved_model.evaluate(
    test_dataset,
    verbose=1,
    return_dict=True
)

improved_test_evaluation_time = (
    time.perf_counter() - test_start_time
)

improved_evaluation_table = pd.DataFrame(
    [
        {
            "Model": "Improved weighted CNN",
            "Subset": "Validation",
            "Loss": validation_results_improved["loss"],
            "MAE": validation_results_improved["mae"],
            "RMSE": validation_results_improved["rmse"]
        },
        {
            "Model": "Improved weighted CNN",
            "Subset": "Test",
            "Loss": test_results_improved["loss"],
            "MAE": test_results_improved["mae"],
            "RMSE": test_results_improved["rmse"]
        }
    ]
)

display(improved_evaluation_table)

print(
    f"Test evaluation time: "
    f"{improved_test_evaluation_time:.2f} seconds"
)

### Calculate Improved-CNN Test Metrics

Generate predictions and compute the same test metrics used for PilotNet, including R-squared and bias. Saving the metrics in a consistent format supports fair comparison.


In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np
import pandas as pd

improved_test_predictions = (
    best_improved_model
    .predict(
        test_dataset,
        verbose=1
    )
    .reshape(-1)
)

assert len(improved_test_predictions) == len(test_actual)

improved_test_mae = mean_absolute_error(
    test_actual,
    improved_test_predictions
)

improved_test_mse = mean_squared_error(
    test_actual,
    improved_test_predictions
)

improved_test_rmse = np.sqrt(
    improved_test_mse
)

improved_test_r2 = r2_score(
    test_actual,
    improved_test_predictions
)

improved_prediction_bias = np.mean(
    improved_test_predictions - test_actual
)

improved_metrics = pd.DataFrame(
    [
        {
            "Model": "Improved weighted CNN",
            "Parameters": best_improved_model.count_params(),
            "Training time (minutes)": (
                improved_training_time / 60
            ),
            "Test MAE": improved_test_mae,
            "Test MSE": improved_test_mse,
            "Test RMSE": improved_test_rmse,
            "Test R2": improved_test_r2,
            "Prediction bias": improved_prediction_bias
        }
    ]
)

display(improved_metrics)

IMPROVED_METRICS_PATH = (
    RESULTS_DIR / "improved_weighted_cnn_metrics.csv"
)

improved_metrics.to_csv(
    IMPROVED_METRICS_PATH,
    index=False
)

print("Metrics saved to:")
print(IMPROVED_METRICS_PATH)

### Analyze Improved-CNN Regional Performance

Measure error and mean prediction within each steering region. This determines whether weighting actually improved turns or merely changed the model's behavior on straight samples.


In [ ]:
improved_prediction_analysis = pd.DataFrame(
    {
        "actual": test_actual,
        "predicted": improved_test_predictions
    }
)

improved_prediction_analysis["absolute_error"] = np.abs(
    improved_prediction_analysis["actual"]
    - improved_prediction_analysis["predicted"]
)

improved_prediction_analysis["steering_region"] = (
    improved_prediction_analysis["actual"]
    .apply(assign_steering_region)
)

improved_region_rows = []

for region in region_order:

    region_data = improved_prediction_analysis[
        improved_prediction_analysis["steering_region"] == region
    ]

    if len(region_data) == 0:
        continue

    improved_region_rows.append(
        {
            "Steering region": region,
            "Test samples": len(region_data),
            "MAE": mean_absolute_error(
                region_data["actual"],
                region_data["predicted"]
            ),
            "RMSE": np.sqrt(
                mean_squared_error(
                    region_data["actual"],
                    region_data["predicted"]
                )
            ),
            "Mean actual": region_data["actual"].mean(),
            "Mean predicted": region_data["predicted"].mean()
        }
    )

improved_region_performance = pd.DataFrame(
    improved_region_rows
)

display(improved_region_performance)

### Compare Overall CNN Results

Place the baseline and improved CNN test metrics side by side and calculate percentage changes in MAE and parameter count.


In [ ]:
baseline_summary = pd.DataFrame(
    [
        {
            "Model": "PilotNet baseline",
            "Parameters": best_pilotnet_model.count_params(),
            "Test MAE": test_mae,
            "Test RMSE": test_rmse,
            "Test R2": test_r2,
            "Prediction bias": test_bias
        }
    ]
)

improved_summary = pd.DataFrame(
    [
        {
            "Model": "Improved weighted CNN",
            "Parameters": best_improved_model.count_params(),
            "Test MAE": improved_test_mae,
            "Test RMSE": improved_test_rmse,
            "Test R2": improved_test_r2,
            "Prediction bias": improved_prediction_bias
        }
    ]
)

cnn_comparison = pd.concat(
    [
        baseline_summary,
        improved_summary
    ],
    ignore_index=True
)

cnn_comparison["MAE improvement vs baseline (%)"] = (
    100
    * (
        test_mae
        - cnn_comparison["Test MAE"]
    )
    / test_mae
)

cnn_comparison["Parameter reduction vs baseline (%)"] = (
    100
    * (
        best_pilotnet_model.count_params()
        - cnn_comparison["Parameters"]
    )
    / best_pilotnet_model.count_params()
)

display(cnn_comparison)

### Compare CNNs by Steering Region

Merge region-level metrics from both models and calculate the MAE improvement for every steering category. Positive and negative percentages show where the architectural changes helped or hurt.


In [ ]:
baseline_region_comparison = (
    region_performance[
        ["Steering region", "MAE", "RMSE", "Mean predicted"]
    ]
    .rename(
        columns={
            "MAE": "PilotNet MAE",
            "RMSE": "PilotNet RMSE",
            "Mean predicted": "PilotNet mean prediction"
        }
    )
)

improved_region_comparison_table = (
    improved_region_performance[
        ["Steering region", "MAE", "RMSE", "Mean predicted"]
    ]
    .rename(
        columns={
            "MAE": "Improved CNN MAE",
            "RMSE": "Improved CNN RMSE",
            "Mean predicted": "Improved CNN mean prediction"
        }
    )
)

region_model_comparison = baseline_region_comparison.merge(
    improved_region_comparison_table,
    on="Steering region"
)

region_model_comparison[
    "MAE improvement (%)"
] = (
    100
    * (
        region_model_comparison["PilotNet MAE"]
        - region_model_comparison["Improved CNN MAE"]
    )
    / region_model_comparison["PilotNet MAE"]
)

display(region_model_comparison)

### Compare CNN Training Histories

Plot training and validation MAE and RMSE for both CNNs. Raw loss curves are not compared because the models use different loss functions, whereas MAE and RMSE are common metrics.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

pilotnet_history_data = pilotnet_history.history
improved_history_data = improved_history.history

pilotnet_epochs = np.arange(
    1,
    len(pilotnet_history_data["mae"]) + 1
)

improved_epochs = np.arange(
    1,
    len(improved_history_data["mae"]) + 1
)

# ------------------------------------------------------------
# Plot 1: Training MAE comparison
# ------------------------------------------------------------
plt.figure(figsize=(10, 5))

plt.plot(
    pilotnet_epochs,
    pilotnet_history_data["mae"],
    marker="o",
    label="PilotNet training MAE"
)

plt.plot(
    improved_epochs,
    improved_history_data["mae"],
    marker="o",
    label="Improved CNN training MAE"
)

plt.xlabel("Epoch")
plt.ylabel("Mean absolute error")
plt.title("Training MAE Comparison")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Plot 2: Validation MAE comparison
# ------------------------------------------------------------
plt.figure(figsize=(10, 5))

plt.plot(
    pilotnet_epochs,
    pilotnet_history_data["val_mae"],
    marker="o",
    label="PilotNet validation MAE"
)

plt.plot(
    improved_epochs,
    improved_history_data["val_mae"],
    marker="o",
    label="Improved CNN validation MAE"
)

plt.xlabel("Epoch")
plt.ylabel("Mean absolute error")
plt.title("Validation MAE Comparison")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Plot 3: Training RMSE comparison
# ------------------------------------------------------------
plt.figure(figsize=(10, 5))

plt.plot(
    pilotnet_epochs,
    pilotnet_history_data["rmse"],
    marker="o",
    label="PilotNet training RMSE"
)

plt.plot(
    improved_epochs,
    improved_history_data["rmse"],
    marker="o",
    label="Improved CNN training RMSE"
)

plt.xlabel("Epoch")
plt.ylabel("Root mean squared error")
plt.title("Training RMSE Comparison")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Plot 4: Validation RMSE comparison
# ------------------------------------------------------------
plt.figure(figsize=(10, 5))

plt.plot(
    pilotnet_epochs,
    pilotnet_history_data["val_rmse"],
    marker="o",
    label="PilotNet validation RMSE"
)

plt.plot(
    improved_epochs,
    improved_history_data["val_rmse"],
    marker="o",
    label="Improved CNN validation RMSE"
)

plt.xlabel("Epoch")
plt.ylabel("Root mean squared error")
plt.title("Validation RMSE Comparison")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### Compare Validation and Test Generalization

Create grouped bar charts for validation and test MAE and RMSE. The gap between validation and test performance indicates how well each model generalizes to the unseen sequence groups.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

model_split_comparison = pd.DataFrame(
    [
        {
            "Model": "PilotNet baseline",
            "Subset": "Validation",
            "MAE": validation_results["mae"],
            "RMSE": validation_results["rmse"]
        },
        {
            "Model": "PilotNet baseline",
            "Subset": "Test",
            "MAE": test_mae,
            "RMSE": test_rmse
        },
        {
            "Model": "Improved weighted CNN",
            "Subset": "Validation",
            "MAE": validation_results_improved["mae"],
            "RMSE": validation_results_improved["rmse"]
        },
        {
            "Model": "Improved weighted CNN",
            "Subset": "Test",
            "MAE": improved_test_mae,
            "RMSE": improved_test_rmse
        }
    ]
)

display(model_split_comparison)

# ------------------------------------------------------------
# Grouped MAE bar chart
# ------------------------------------------------------------
mae_pivot = model_split_comparison.pivot(
    index="Model",
    columns="Subset",
    values="MAE"
)

mae_pivot.plot(
    kind="bar",
    figsize=(10, 5)
)

plt.ylabel("Mean absolute error")
plt.title("Validation and Test MAE Comparison")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Grouped RMSE bar chart
# ------------------------------------------------------------
rmse_pivot = model_split_comparison.pivot(
    index="Model",
    columns="Subset",
    values="RMSE"
)

rmse_pivot.plot(
    kind="bar",
    figsize=(10, 5)
)

plt.ylabel("Root mean squared error")
plt.title("Validation and Test RMSE Comparison")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

### Compare Sequential Steering Predictions

Overlay actual steering with predictions from both CNNs for an ordered portion of the test set. This illustrates responsiveness, smoothing, and underprediction during turns.


In [ ]:
number_to_plot = min(
    300,
    len(test_actual)
)

plt.figure(figsize=(15, 6))

plt.plot(
    test_actual[:number_to_plot],
    label="Actual steering",
    linewidth=2
)

plt.plot(
    test_predictions[:number_to_plot],
    label="PilotNet prediction",
    linewidth=1.3
)

plt.plot(
    improved_test_predictions[:number_to_plot],
    label="Improved CNN prediction",
    linewidth=1.3
)

plt.xlabel("Test sample")
plt.ylabel("Steering angle")
plt.title(
    f"Actual and Predicted Steering for First "
    f"{number_to_plot} Test Samples"
)

plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### Compare Actual-versus-Predicted Scatter

Create separate scatter plots for PilotNet and the improved CNN against the ideal diagonal. Compression toward the horizontal center indicates regression toward zero.


In [ ]:
minimum_value = min(
    test_actual.min(),
    test_predictions.min(),
    improved_test_predictions.min()
)

maximum_value = max(
    test_actual.max(),
    test_predictions.max(),
    improved_test_predictions.max()
)

# PilotNet
plt.figure(figsize=(7, 7))

plt.scatter(
    test_actual,
    test_predictions,
    alpha=0.4,
    s=18
)

plt.plot(
    [minimum_value, maximum_value],
    [minimum_value, maximum_value],
    linestyle="--",
    label="Ideal prediction"
)

plt.xlabel("Actual steering")
plt.ylabel("Predicted steering")
plt.title("PilotNet: Actual versus Predicted")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# Improved CNN
plt.figure(figsize=(7, 7))

plt.scatter(
    test_actual,
    improved_test_predictions,
    alpha=0.4,
    s=18
)

plt.plot(
    [minimum_value, maximum_value],
    [minimum_value, maximum_value],
    linestyle="--",
    label="Ideal prediction"
)

plt.xlabel("Actual steering")
plt.ylabel("Predicted steering")
plt.title("Improved CNN: Actual versus Predicted")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### Comments
The improved model is not clearly superior overall.

It achieved:

*  Slightly better test MAE: 0.09376 vs 0.09620
*  Worse test RMSE: 0.15034 vs 0.13646
*  Worse R2−0.039 vs 0.144
*  Much better straight-driving MAE
*  Substantially worse left-turn and sharp-left performance

This means it became more specialized toward predicting near-straight steering. The positive prediction bias of 0.0446 also shows a systematic shift toward positive steering.

Therefore, at this stage, PilotNet remains the more reliable overall model, despite the improved CNN having lower average MAE and fewer parameters.

### Define MobileNetV2 Preprocessing

Crop and resize road images for a 160 by 96 RGB input, then apply the normalization expected by ImageNet-pretrained MobileNetV2. This pipeline is intentionally different from the YUV PilotNet pipeline.


In [ ]:
# ============================================================
# CORRECTED BLOCK 44: DEFINE MOBILENETV2 IMAGE PREPROCESSING
# ============================================================

import cv2
import numpy as np
import tensorflow as tf

TRANSFER_HEIGHT = 96
TRANSFER_WIDTH = 160
TRANSFER_CHANNELS = 3

def preprocess_transfer_image(image):
    """
    Preprocess an RGB road image for MobileNetV2.

    Steps:
    1. Crop sky and vehicle hood.
    2. Resize to 160 x 96.
    3. Keep the image in RGB.
    4. Apply MobileNetV2 preprocessing.
    """

    image_height = image.shape[0]

    cropped_image = image[
        CROP_TOP : image_height - CROP_BOTTOM,
        :,
        :
    ]

    resized_image = cv2.resize(
        cropped_image,
        (TRANSFER_WIDTH, TRANSFER_HEIGHT),
        interpolation=cv2.INTER_AREA
    )

    resized_image = resized_image.astype(np.float32)

    processed_image = (
        tf.keras.applications.mobilenet_v2.preprocess_input(
            resized_image
        )
    )

    return np.asarray(
        processed_image,
        dtype=np.float32
    )

### Verify MobileNetV2 Inputs

Display an original image beside the processed MobileNetV2 input and report its shape and range. This confirms that transfer-learning inputs match the pretrained network's expectations.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

transfer_sample = train_samples.sample(
    n=1,
    random_state=SEED
).iloc[0]

transfer_original = load_image_rgb(
    transfer_sample["image_path"]
)

transfer_processed = preprocess_transfer_image(
    transfer_original
)

# Reverse MobileNetV2 preprocessing for display.
transfer_display = np.clip(
    (transfer_processed + 1.0) * 127.5,
    0,
    255
).astype(np.uint8)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 4)
)

axes[0].imshow(transfer_original)
axes[0].set_title(
    f"Original image\n{transfer_original.shape}"
)
axes[0].axis("off")

axes[1].imshow(transfer_display)
axes[1].set_title(
    f"MobileNetV2 input\n{transfer_processed.shape}"
)
axes[1].axis("off")

plt.tight_layout()
plt.show()

print("Processed shape:", transfer_processed.shape)
print("Processed dtype:", transfer_processed.dtype)
print("Minimum value:", transfer_processed.min())
print("Maximum value:", transfer_processed.max())

### Create Transfer-Learning Input Functions

Define separate image loaders for augmented training samples and deterministic evaluation samples. The TensorFlow wrappers preserve static tensor shapes after using NumPy and OpenCV operations.


In [ ]:
def load_and_process_transfer_training_sample(
    path_bytes,
    steering_value
):
    """
    Load, augment, and preprocess a training image for MobileNetV2.
    """

    image_path = path_bytes.decode("utf-8")
    steering = float(steering_value)

    image = load_image_rgb(image_path)

    rng = np.random.default_rng()

    image, steering = augment_image(
        image,
        steering,
        rng
    )

    image = preprocess_transfer_image(image)

    return (
        image.astype(np.float32),
        np.float32(steering)
    )


def load_and_process_transfer_evaluation_sample(
    path_bytes,
    steering_value
):
    """
    Load and preprocess validation or test data without augmentation.
    """

    image_path = path_bytes.decode("utf-8")
    steering = float(steering_value)

    image = load_image_rgb(image_path)
    image = preprocess_transfer_image(image)

    return (
        image.astype(np.float32),
        np.float32(steering)
    )


def tf_process_transfer_training_sample(
    image_path,
    steering
):
    image, steering = tf.numpy_function(
        func=load_and_process_transfer_training_sample,
        inp=[image_path, steering],
        Tout=[tf.float32, tf.float32]
    )

    image.set_shape(
        [
            TRANSFER_HEIGHT,
            TRANSFER_WIDTH,
            TRANSFER_CHANNELS
        ]
    )

    steering.set_shape([])

    return image, steering


def tf_process_transfer_evaluation_sample(
    image_path,
    steering
):
    image, steering = tf.numpy_function(
        func=load_and_process_transfer_evaluation_sample,
        inp=[image_path, steering],
        Tout=[tf.float32, tf.float32]
    )

    image.set_shape(
        [
            TRANSFER_HEIGHT,
            TRANSFER_WIDTH,
            TRANSFER_CHANNELS
        ]
    )

    steering.set_shape([])

    return image, steering


def create_transfer_dataset(
    samples_dataframe,
    training=False,
    batch_size=64
):
    image_paths = (
        samples_dataframe["image_path"]
        .astype(str)
        .to_numpy()
    )

    steering_values = (
        samples_dataframe["steering"]
        .astype(np.float32)
        .to_numpy()
    )

    dataset = tf.data.Dataset.from_tensor_slices(
        (image_paths, steering_values)
    )

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(samples_dataframe),
            seed=SEED,
            reshuffle_each_iteration=True
        )

        dataset = dataset.map(
            tf_process_transfer_training_sample,
            num_parallel_calls=AUTOTUNE
        )

    else:
        dataset = dataset.map(
            tf_process_transfer_evaluation_sample,
            num_parallel_calls=AUTOTUNE
        )

    dataset = dataset.batch(
        batch_size,
        drop_remainder=False
    )

    dataset = dataset.prefetch(AUTOTUNE)

    return dataset

### Build MobileNetV2 Datasets

Create training, validation, and test pipelines with a smaller batch size suitable for the larger pretrained model. A sample batch is checked before training.


In [ ]:
# A smaller batch is safer for the larger transfer-learning model.
TRANSFER_BATCH_SIZE = 32

transfer_train_dataset = create_transfer_dataset(
    train_samples,
    training=True,
    batch_size=TRANSFER_BATCH_SIZE
)

transfer_validation_dataset = create_transfer_dataset(
    validation_samples,
    training=False,
    batch_size=TRANSFER_BATCH_SIZE
)

transfer_test_dataset = create_transfer_dataset(
    test_samples,
    training=False,
    batch_size=TRANSFER_BATCH_SIZE
)

transfer_images, transfer_labels = next(
    iter(transfer_train_dataset)
)

print("Transfer image batch shape:", transfer_images.shape)
print("Transfer label batch shape:", transfer_labels.shape)

print(
    "Transfer image range:",
    float(tf.reduce_min(transfer_images)),
    "to",
    float(tf.reduce_max(transfer_images))
)

### Define the Frozen MobileNetV2 Regressor

Load an ImageNet-pretrained MobileNetV2 backbone without its classification head, freeze it, and attach global pooling plus dense regression layers. Only the new steering head is trainable in this stage.


In [ ]:
from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input,
    GlobalAveragePooling2D,
    Dense,
    Dropout
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

def build_mobilenet_regressor(
    learning_rate=3e-4,
    dropout_rate=0.30,
    l2_strength=1e-5
):
    inputs = Input(
        shape=(
            TRANSFER_HEIGHT,
            TRANSFER_WIDTH,
            TRANSFER_CHANNELS
        ),
        name="road_image"
    )

    feature_extractor = (
        tf.keras.applications.MobileNetV2(
            input_shape=(
                TRANSFER_HEIGHT,
                TRANSFER_WIDTH,
                TRANSFER_CHANNELS
            ),
            include_top=False,
            weights="imagenet"
        )
    )

    # Freeze all pretrained layers during the first stage.
    feature_extractor.trainable = False

    # training=False keeps MobileNetV2 batch-normalization layers
    # in inference mode while the backbone is frozen.
    x = feature_extractor(
        inputs,
        training=False
    )

    x = GlobalAveragePooling2D(
        name="global_average_pooling"
    )(x)

    x = Dense(
        128,
        activation="relu",
        kernel_regularizer=l2(l2_strength),
        name="regression_dense_128"
    )(x)

    x = Dropout(
        dropout_rate,
        name="regression_dropout"
    )(x)

    x = Dense(
        32,
        activation="relu",
        kernel_regularizer=l2(l2_strength),
        name="regression_dense_32"
    )(x)

    outputs = Dense(
        1,
        activation="linear",
        name="steering_output"
    )(x)

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name="MobileNetV2_Steering_Regressor"
    )

    model.compile(
        optimizer=Adam(
            learning_rate=learning_rate
        ),
        loss=tf.keras.losses.Huber(
            delta=0.15
        ),
        metrics=[
            tf.keras.metrics.MeanAbsoluteError(
                name="mae"
            ),
            tf.keras.metrics.RootMeanSquaredError(
                name="rmse"
            )
        ]
    )

    return model


mobilenet_model = build_mobilenet_regressor()

mobilenet_model.summary()

print(
    "\nTotal parameters:",
    mobilenet_model.count_params()
)

print(
    "Trainable parameters:",
    sum(
        np.prod(variable.shape)
        for variable in mobilenet_model.trainable_weights
    )
)

print(
    "Frozen parameters:",
    sum(
        np.prod(variable.shape)
        for variable in mobilenet_model.non_trainable_weights
    )
)

### Configure MobileNetV2 Callbacks

Specify checkpoint, early-stopping, learning-rate, logging, and numerical-safety callbacks for the frozen-backbone experiment.


In [ ]:
MOBILENET_MODEL_PATH = (
    MODEL_DIR / "mobilenetv2_frozen_best.keras"
)

MOBILENET_LOG_PATH = (
    RESULTS_DIR / "mobilenetv2_frozen_training.csv"
)

mobilenet_callbacks = [
    ModelCheckpoint(
        filepath=str(MOBILENET_MODEL_PATH),
        monitor="val_mae",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    EarlyStopping(
        monitor="val_mae",
        mode="min",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_mae",
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),

    CSVLogger(
        filename=str(MOBILENET_LOG_PATH),
        append=False
    ),

    TerminateOnNaN()
]

print("MobileNetV2 checkpoint:")
print(MOBILENET_MODEL_PATH)

### Train the Frozen MobileNetV2 Model

Train only the regression head while the pretrained visual backbone remains fixed. This tests whether generic ImageNet features transfer to steering prediction with limited computation.


In [ ]:
import time
import numpy as np

MOBILENET_MAX_EPOCHS = 15

mobilenet_start_time = time.perf_counter()

mobilenet_history = mobilenet_model.fit(
    transfer_train_dataset,
    validation_data=transfer_validation_dataset,
    epochs=MOBILENET_MAX_EPOCHS,
    callbacks=mobilenet_callbacks,
    verbose=1
)

mobilenet_training_time = (
    time.perf_counter() - mobilenet_start_time
)

mobilenet_best_epoch = (
    int(
        np.argmin(
            mobilenet_history.history["val_mae"]
        )
    ) + 1
)

print(
    f"\nMobileNetV2 training time: "
    f"{mobilenet_training_time / 60:.2f} minutes"
)

print(
    "Completed epochs:",
    len(mobilenet_history.history["loss"])
)

print(
    "Best epoch:",
    mobilenet_best_epoch
)

print(
    "Best validation MAE:",
    mobilenet_history.history["val_mae"][
        mobilenet_best_epoch - 1
    ]
)

print(
    "Validation RMSE at best epoch:",
    mobilenet_history.history["val_rmse"][
        mobilenet_best_epoch - 1
    ]
)

### Plot MobileNetV2 Learning Curves

Plot training and validation MAE and RMSE and identify the best validation epoch. These curves help assess convergence and overfitting of the regression head.


In [ ]:
mobilenet_epochs = np.arange(
    1,
    len(mobilenet_history.history["mae"]) + 1
)

plt.figure(figsize=(10, 5))

plt.plot(
    mobilenet_epochs,
    mobilenet_history.history["mae"],
    marker="o",
    label="Training MAE"
)

plt.plot(
    mobilenet_epochs,
    mobilenet_history.history["val_mae"],
    marker="o",
    label="Validation MAE"
)

plt.xlabel("Epoch")
plt.ylabel("Mean absolute error")
plt.title("Frozen MobileNetV2 Training History")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


plt.figure(figsize=(10, 5))

plt.plot(
    mobilenet_epochs,
    mobilenet_history.history["rmse"],
    marker="o",
    label="Training RMSE"
)

plt.plot(
    mobilenet_epochs,
    mobilenet_history.history["val_rmse"],
    marker="o",
    label="Validation RMSE"
)

plt.xlabel("Epoch")
plt.ylabel("Root mean squared error")
plt.title("Frozen MobileNetV2 RMSE History")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### Comments
MobileNetV2 reduces validation MAE by about 32.3% relative to PilotNet and 24.5% relative to the improved CNN. Its RMSE improvement is smaller, suggesting some larger errors remain, but the result clearly justifies a brief fine-tuning stage. We will fine-tune only the upper MobileNetV2 layers using a very small learning rate. We will select between the frozen and fine-tuned checkpoints using validation MAE, then evaluate only the selected version on the test set.

### Load the Frozen Checkpoint and Find the Backbone

Reload the best frozen model and locate the nested MobileNetV2 feature extractor. The layer listing confirms which top-level component will be partially unfrozen.


In [ ]:
import tensorflow as tf
import numpy as np

frozen_mobilenet_model = tf.keras.models.load_model(
    str(MOBILENET_MODEL_PATH)
)

print("Frozen MobileNetV2 checkpoint loaded.")
print("Total parameters:", frozen_mobilenet_model.count_params())

# Display top-level layers so we can identify the feature extractor.
for index, layer in enumerate(frozen_mobilenet_model.layers):
    print(
        f"{index:2d} | "
        f"{layer.name:35s} | "
        f"{layer.__class__.__name__:25s} | "
        f"trainable={layer.trainable}"
    )

# The nested MobileNetV2 model is the layer whose class is Functional
# and whose name normally contains "mobilenet".
mobilenet_base = None

for layer in frozen_mobilenet_model.layers:
    if (
        isinstance(layer, tf.keras.Model)
        and "mobilenet" in layer.name.lower()
    ):
        mobilenet_base = layer
        break

if mobilenet_base is None:
    raise ValueError(
        "The nested MobileNetV2 feature extractor could not be found."
    )

print("\nDetected feature extractor:")
print(mobilenet_base.name)

print("Number of backbone layers:")
print(len(mobilenet_base.layers))

### Configure Partial Fine-Tuning

Unfreeze only the final backbone layers while keeping batch-normalization layers frozen. This allows high-level features to adapt to road geometry without destabilizing the entire pretrained network.


In [ ]:
from tensorflow.keras.layers import BatchNormalization

# Make the backbone eligible for partial fine-tuning.
mobilenet_base.trainable = True

# Fine-tune approximately the final 25 layers.
FINE_TUNE_LAST_N_LAYERS = 25

fine_tune_start = max(
    0,
    len(mobilenet_base.layers) - FINE_TUNE_LAST_N_LAYERS
)

for index, layer in enumerate(mobilenet_base.layers):

    if index < fine_tune_start:
        layer.trainable = False

    else:
        # Keep all BatchNormalization layers frozen.
        if isinstance(layer, BatchNormalization):
            layer.trainable = False
        else:
            layer.trainable = True

trainable_backbone_layers = [
    layer.name
    for layer in mobilenet_base.layers
    if layer.trainable
]

print("Fine-tuning begins at backbone layer index:")
print(fine_tune_start)

print("\nTrainable backbone layers:")
for layer_name in trainable_backbone_layers:
    print(layer_name)

trainable_parameters = sum(
    np.prod(variable.shape)
    for variable in frozen_mobilenet_model.trainable_weights
)

non_trainable_parameters = sum(
    np.prod(variable.shape)
    for variable in frozen_mobilenet_model.non_trainable_weights
)

print("\nTrainable parameters after partial unfreezing:")
print(trainable_parameters)

print("Non-trainable parameters:")
print(non_trainable_parameters)

### Recompile for Fine-Tuning

Compile the partially trainable model with a much smaller learning rate. Recompilation is required after changing layer trainability.


In [ ]:
from tensorflow.keras.optimizers import Adam

FINE_TUNE_LEARNING_RATE = 1e-5

frozen_mobilenet_model.compile(
    optimizer=Adam(
        learning_rate=FINE_TUNE_LEARNING_RATE
    ),
    loss=tf.keras.losses.Huber(
        delta=0.15
    ),
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(
            name="mae"
        ),
        tf.keras.metrics.RootMeanSquaredError(
            name="rmse"
        )
    ]
)

print("Model recompiled for fine-tuning.")
print("Learning rate:", FINE_TUNE_LEARNING_RATE)

### Configure Fine-Tuning Callbacks

Create a separate checkpoint and log for fine-tuning so the original frozen model is preserved for direct validation comparison.


In [ ]:
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau,
    CSVLogger,
    TerminateOnNaN
)

MOBILENET_FINETUNED_PATH = (
    MODEL_DIR / "mobilenetv2_finetuned_best.keras"
)

MOBILENET_FINETUNE_LOG_PATH = (
    RESULTS_DIR / "mobilenetv2_finetuning.csv"
)

fine_tuning_callbacks = [
    ModelCheckpoint(
        filepath=str(MOBILENET_FINETUNED_PATH),
        monitor="val_mae",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    EarlyStopping(
        monitor="val_mae",
        mode="min",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_mae",
        mode="min",
        factor=0.3,
        patience=1,
        min_lr=1e-7,
        verbose=1
    ),

    CSVLogger(
        filename=str(MOBILENET_FINETUNE_LOG_PATH),
        append=False
    ),

    TerminateOnNaN()
]

print("Fine-tuned checkpoint will be saved to:")
print(MOBILENET_FINETUNED_PATH)

### Fine-Tune the Upper MobileNetV2 Layers

Continue training for a small number of epochs with the low learning rate and record the best fine-tuned validation metrics.


In [ ]:
import time
import numpy as np

FINE_TUNE_MAX_EPOCHS = 8

fine_tuning_start_time = time.perf_counter()

mobilenet_finetune_history = frozen_mobilenet_model.fit(
    transfer_train_dataset,
    validation_data=transfer_validation_dataset,
    epochs=FINE_TUNE_MAX_EPOCHS,
    callbacks=fine_tuning_callbacks,
    verbose=1
)

mobilenet_finetuning_time = (
    time.perf_counter() - fine_tuning_start_time
)

fine_tuning_best_epoch = (
    int(
        np.argmin(
            mobilenet_finetune_history.history["val_mae"]
        )
    ) + 1
)

fine_tuned_best_val_mae = (
    mobilenet_finetune_history.history["val_mae"][
        fine_tuning_best_epoch - 1
    ]
)

fine_tuned_best_val_rmse = (
    mobilenet_finetune_history.history["val_rmse"][
        fine_tuning_best_epoch - 1
    ]
)

print(
    f"\nFine-tuning time: "
    f"{mobilenet_finetuning_time / 60:.2f} minutes"
)

print(
    "Completed fine-tuning epochs:",
    len(mobilenet_finetune_history.history["loss"])
)

print("Best fine-tuning epoch:", fine_tuning_best_epoch)
print("Best fine-tuned validation MAE:", fine_tuned_best_val_mae)
print("Fine-tuned validation RMSE:", fine_tuned_best_val_rmse)

### Select Frozen or Fine-Tuned MobileNetV2

Compare the two training stages on validation MAE and RMSE, select the checkpoint with lower validation MAE, and load it for final testing.


In [ ]:
import pandas as pd
import tensorflow as tf

frozen_best_val_mae = min(
    mobilenet_history.history["val_mae"]
)

frozen_best_epoch = (
    int(
        np.argmin(
            mobilenet_history.history["val_mae"]
        )
    ) + 1
)

frozen_best_val_rmse = (
    mobilenet_history.history["val_rmse"][
        frozen_best_epoch - 1
    ]
)

mobilenet_stage_comparison = pd.DataFrame(
    [
        {
            "Stage": "Frozen backbone",
            "Best epoch": frozen_best_epoch,
            "Validation MAE": frozen_best_val_mae,
            "Validation RMSE": frozen_best_val_rmse,
            "Training time (minutes)": (
                mobilenet_training_time / 60
            )
        },
        {
            "Stage": "Partially fine-tuned",
            "Best epoch": fine_tuning_best_epoch,
            "Validation MAE": fine_tuned_best_val_mae,
            "Validation RMSE": fine_tuned_best_val_rmse,
            "Training time (minutes)": (
                mobilenet_finetuning_time / 60
            )
        }
    ]
)

display(mobilenet_stage_comparison)

if fine_tuned_best_val_mae < frozen_best_val_mae:

    selected_mobilenet_path = MOBILENET_FINETUNED_PATH
    selected_mobilenet_stage = "Partially fine-tuned MobileNetV2"

else:

    selected_mobilenet_path = MOBILENET_MODEL_PATH
    selected_mobilenet_stage = "Frozen MobileNetV2"

selected_mobilenet_model = tf.keras.models.load_model(
    str(selected_mobilenet_path)
)

print("\nSelected model:")
print(selected_mobilenet_stage)

print("Selected checkpoint:")
print(selected_mobilenet_path)

### Evaluate Selected MobileNetV2

Measure validation and test performance for the chosen MobileNetV2 stage and record evaluation time.


In [ ]:
import time
import pandas as pd

mobilenet_validation_results = (
    selected_mobilenet_model.evaluate(
        transfer_validation_dataset,
        verbose=1,
        return_dict=True
    )
)

mobilenet_test_start_time = time.perf_counter()

mobilenet_test_results = (
    selected_mobilenet_model.evaluate(
        transfer_test_dataset,
        verbose=1,
        return_dict=True
    )
)

mobilenet_test_evaluation_time = (
    time.perf_counter() - mobilenet_test_start_time
)

mobilenet_evaluation_table = pd.DataFrame(
    [
        {
            "Model": selected_mobilenet_stage,
            "Subset": "Validation",
            "Loss": mobilenet_validation_results["loss"],
            "MAE": mobilenet_validation_results["mae"],
            "RMSE": mobilenet_validation_results["rmse"]
        },
        {
            "Model": selected_mobilenet_stage,
            "Subset": "Test",
            "Loss": mobilenet_test_results["loss"],
            "MAE": mobilenet_test_results["mae"],
            "RMSE": mobilenet_test_results["rmse"]
        }
    ]
)

display(mobilenet_evaluation_table)

print(
    f"Test evaluation time: "
    f"{mobilenet_test_evaluation_time:.2f} seconds"
)

### Calculate Complete MobileNetV2 Metrics

Generate test predictions and calculate MAE, MSE, RMSE, R-squared, bias, parameter counts, and accumulated training time.


In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np
import pandas as pd

mobilenet_test_predictions = (
    selected_mobilenet_model.predict(
        transfer_test_dataset,
        verbose=1
    )
    .reshape(-1)
)

assert len(mobilenet_test_predictions) == len(test_actual)

mobilenet_test_mae = mean_absolute_error(
    test_actual,
    mobilenet_test_predictions
)

mobilenet_test_mse = mean_squared_error(
    test_actual,
    mobilenet_test_predictions
)

mobilenet_test_rmse = np.sqrt(
    mobilenet_test_mse
)

mobilenet_test_r2 = r2_score(
    test_actual,
    mobilenet_test_predictions
)

mobilenet_prediction_bias = np.mean(
    mobilenet_test_predictions - test_actual
)

mobilenet_total_training_time = (
    mobilenet_training_time
)

if selected_mobilenet_stage == "Partially fine-tuned MobileNetV2":
    mobilenet_total_training_time += (
        mobilenet_finetuning_time
    )

mobilenet_metrics = pd.DataFrame(
    [
        {
            "Model": selected_mobilenet_stage,
            "Parameters": selected_mobilenet_model.count_params(),
            "Trainable parameters during selected stage": (
                168129
                if selected_mobilenet_stage == "Frozen MobileNetV2"
                else sum(
                    np.prod(variable.shape)
                    for variable
                    in selected_mobilenet_model.trainable_weights
                )
            ),
            "Training time (minutes)": (
                mobilenet_total_training_time / 60
            ),
            "Test MAE": mobilenet_test_mae,
            "Test MSE": mobilenet_test_mse,
            "Test RMSE": mobilenet_test_rmse,
            "Test R2": mobilenet_test_r2,
            "Prediction bias": mobilenet_prediction_bias
        }
    ]
)

display(mobilenet_metrics)

MOBILENET_METRICS_PATH = (
    RESULTS_DIR / "mobilenetv2_metrics.csv"
)

mobilenet_metrics.to_csv(
    MOBILENET_METRICS_PATH,
    index=False
)

print("Metrics saved to:")
print(MOBILENET_METRICS_PATH)

### Analyze MobileNetV2 by Steering Region

Calculate region-specific errors and compare mean predictions with mean true steering. This reveals whether low overall MAE comes from excellent straight-driving performance or genuinely accurate turning.


In [ ]:
mobilenet_prediction_analysis = pd.DataFrame(
    {
        "actual": test_actual,
        "predicted": mobilenet_test_predictions
    }
)

mobilenet_prediction_analysis["absolute_error"] = np.abs(
    mobilenet_prediction_analysis["actual"]
    - mobilenet_prediction_analysis["predicted"]
)

mobilenet_prediction_analysis["steering_region"] = (
    mobilenet_prediction_analysis["actual"]
    .apply(assign_steering_region)
)

mobilenet_region_rows = []

for region in region_order:

    region_data = mobilenet_prediction_analysis[
        mobilenet_prediction_analysis[
            "steering_region"
        ] == region
    ]

    if len(region_data) == 0:
        continue

    mobilenet_region_rows.append(
        {
            "Steering region": region,
            "Test samples": len(region_data),
            "MAE": mean_absolute_error(
                region_data["actual"],
                region_data["predicted"]
            ),
            "RMSE": np.sqrt(
                mean_squared_error(
                    region_data["actual"],
                    region_data["predicted"]
                )
            ),
            "Mean actual": region_data["actual"].mean(),
            "Mean predicted": region_data["predicted"].mean()
        }
    )

mobilenet_region_performance = pd.DataFrame(
    mobilenet_region_rows
)

display(mobilenet_region_performance)

### Compare the First Three Models

Assemble PilotNet, improved CNN, and MobileNetV2 metrics into one table and calculate test-MAE improvement relative to the baseline.


In [ ]:
# ============================================================
# BLOCK 60: OVERALL COMPARISON OF ALL THREE MODELS
# ============================================================

all_model_comparison = pd.DataFrame(
    [
        {
            "Model": "PilotNet baseline",
            "Parameters": best_pilotnet_model.count_params(),
            "Validation MAE": validation_results["mae"],
            "Validation RMSE": validation_results["rmse"],
            "Test MAE": test_mae,
            "Test RMSE": test_rmse,
            "Test R2": test_r2,
            "Prediction bias": test_bias
        },
        {
            "Model": "Improved weighted CNN",
            "Parameters": best_improved_model.count_params(),
            "Validation MAE": validation_results_improved["mae"],
            "Validation RMSE": validation_results_improved["rmse"],
            "Test MAE": improved_test_mae,
            "Test RMSE": improved_test_rmse,
            "Test R2": improved_test_r2,
            "Prediction bias": improved_prediction_bias
        },
        {
            "Model": selected_mobilenet_stage,
            "Parameters": selected_mobilenet_model.count_params(),
            "Validation MAE": mobilenet_validation_results["mae"],
            "Validation RMSE": mobilenet_validation_results["rmse"],
            "Test MAE": mobilenet_test_mae,
            "Test RMSE": mobilenet_test_rmse,
            "Test R2": mobilenet_test_r2,
            "Prediction bias": mobilenet_prediction_bias
        }
    ]
)

all_model_comparison[
    "Test MAE improvement vs PilotNet (%)"
] = (
    100
    * (
        test_mae
        - all_model_comparison["Test MAE"]
    )
    / test_mae
)

display(
    all_model_comparison.sort_values(
        "Test MAE"
    ).reset_index(drop=True)
)

### Plot Three-Model Validation and Test Metrics

Visualize MAE and RMSE across the three conventional image models to compare average accuracy and sensitivity to large errors.


In [ ]:
# ============================================================
# BLOCK 61: PLOT VALIDATION AND TEST METRICS FOR ALL MODELS
# ============================================================

import matplotlib.pyplot as plt

comparison_for_plot = (
    all_model_comparison.set_index("Model")
)

comparison_for_plot[
    ["Validation MAE", "Test MAE"]
].plot(
    kind="bar",
    figsize=(11, 5)
)

plt.ylabel("Mean absolute error")
plt.title("Validation and Test MAE Across Models")
plt.xticks(rotation=15)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


comparison_for_plot[
    ["Validation RMSE", "Test RMSE"]
].plot(
    kind="bar",
    figsize=(11, 5)
)

plt.ylabel("Root mean squared error")
plt.title("Validation and Test RMSE Across Models")
plt.xticks(rotation=15)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

### Compare Three Models by Steering Region

Merge regional MAE tables and display a grouped bar chart showing how each architecture behaves from sharp-left through sharp-right cases.


In [ ]:
# ============================================================
# BLOCK 62: COMPARE STEERING-REGION MAE ACROSS MODELS
# ============================================================

pilotnet_region_mae = (
    region_performance[
        ["Steering region", "MAE"]
    ]
    .rename(
        columns={
            "MAE": "PilotNet MAE"
        }
    )
)

improved_region_mae = (
    improved_region_performance[
        ["Steering region", "MAE"]
    ]
    .rename(
        columns={
            "MAE": "Improved CNN MAE"
        }
    )
)

mobilenet_region_mae = (
    mobilenet_region_performance[
        ["Steering region", "MAE"]
    ]
    .rename(
        columns={
            "MAE": "MobileNetV2 MAE"
        }
    )
)

all_region_comparison = (
    pilotnet_region_mae
    .merge(
        improved_region_mae,
        on="Steering region"
    )
    .merge(
        mobilenet_region_mae,
        on="Steering region"
    )
)

all_region_comparison["Steering region"] = pd.Categorical(
    all_region_comparison["Steering region"],
    categories=region_order,
    ordered=True
)

all_region_comparison = (
    all_region_comparison
    .sort_values("Steering region")
    .reset_index(drop=True)
)

display(all_region_comparison)

all_region_comparison.set_index(
    "Steering region"
).plot(
    kind="bar",
    figsize=(12, 6)
)

plt.ylabel("Mean absolute error")
plt.title("Test MAE by Steering Region and Model")
plt.xticks(rotation=20)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

### Compare Three Models on a Test Sequence

Overlay their steering predictions on the same ordered test segment to inspect tracking behavior and turn-magnitude compression.


In [ ]:
# ============================================================
# BLOCK 63: COMPARE TEST-SEQUENCE PREDICTIONS
# ============================================================

number_to_plot = min(
    300,
    len(test_actual)
)

plt.figure(figsize=(15, 6))

plt.plot(
    test_actual[:number_to_plot],
    label="Actual steering",
    linewidth=2
)

plt.plot(
    test_predictions[:number_to_plot],
    label="PilotNet",
    linewidth=1.1
)

plt.plot(
    improved_test_predictions[:number_to_plot],
    label="Improved CNN",
    linewidth=1.1
)

plt.plot(
    mobilenet_test_predictions[:number_to_plot],
    label="MobileNetV2",
    linewidth=1.2
)

plt.xlabel("Test sample")
plt.ylabel("Steering angle")
plt.title(
    f"Steering Predictions for First "
    f"{number_to_plot} Test Samples"
)

plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### Install and Import CLIP Dependencies

Install Hugging Face tooling, import PyTorch and CLIP classes, and select a CUDA device when available. The CLIP encoder is separate from the TensorFlow regression head.


In [ ]:
# ============================================================
# BLOCK 64: INSTALL AND IMPORT CLIP DEPENDENCIES
# ============================================================

!pip install -q transformers accelerate

import os
import time
import numpy as np
import pandas as pd
import torch

from PIL import Image
from torch.utils.data import Dataset, DataLoader

from transformers import (
    CLIPImageProcessor,
    CLIPVisionModelWithProjection
)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Selected device:", DEVICE)

### Load the Frozen CLIP Vision Encoder

Download `CLIP-ViT-B/32`, move it to the selected device, switch to evaluation mode, and freeze every parameter. The encoder supplies vision-language-pretrained image representations.


In [ ]:
# ============================================================
# BLOCK 65: LOAD PRETRAINED CLIP VISION ENCODER
# ============================================================

CLIP_MODEL_ID = "openai/clip-vit-base-patch32"

clip_processor = CLIPImageProcessor.from_pretrained(
    CLIP_MODEL_ID
)

clip_vision_model = (
    CLIPVisionModelWithProjection.from_pretrained(
        CLIP_MODEL_ID
    )
)

clip_vision_model = clip_vision_model.to(DEVICE)
clip_vision_model.eval()

# Freeze all CLIP parameters.
for parameter in clip_vision_model.parameters():
    parameter.requires_grad = False

print("CLIP vision model loaded successfully.")

print(
    "Total CLIP parameters:",
    sum(
        parameter.numel()
        for parameter in clip_vision_model.parameters()
    )
)

print(
    "Trainable CLIP parameters:",
    sum(
        parameter.numel()
        for parameter in clip_vision_model.parameters()
        if parameter.requires_grad
    )
)

### Define the CLIP Steering Dataset

Load and crop each road image, convert it to a PIL image, and let the CLIP processor resize and normalize it. Each item returns CLIP pixel values and the continuous steering target.


In [ ]:
# ============================================================
# BLOCK 66: DEFINE CLIP IMAGE DATASET
# ============================================================

class SteeringCLIPDataset(Dataset):
    """
    Dataset that loads road images and steering labels for
    frozen CLIP feature extraction.
    """

    def __init__(
        self,
        samples_dataframe,
        processor
    ):
        self.samples = (
            samples_dataframe
            .reset_index(drop=True)
            .copy()
        )

        self.processor = processor

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):

        row = self.samples.iloc[index]

        image = load_image_rgb(
            row["image_path"]
        )

        image_height = image.shape[0]

        # Use the same road-focused crop.
        image = image[
            CROP_TOP : image_height - CROP_BOTTOM,
            :,
            :
        ]

        image = Image.fromarray(image)

        processed = self.processor(
            images=image,
            return_tensors="pt"
        )

        pixel_values = processed[
            "pixel_values"
        ].squeeze(0)

        steering = torch.tensor(
            float(row["steering"]),
            dtype=torch.float32
        )

        return pixel_values, steering

### Create CLIP DataLoaders

Build deterministic PyTorch loaders for training, validation, and test feature extraction. Shuffling is disabled so embeddings remain aligned with steering labels.


In [ ]:
# ============================================================
# BLOCK 67: CREATE CLIP FEATURE EXTRACTION LOADERS
# ============================================================

CLIP_BATCH_SIZE = 64

clip_train_dataset = SteeringCLIPDataset(
    train_samples,
    clip_processor
)

clip_validation_dataset = SteeringCLIPDataset(
    validation_samples,
    clip_processor
)

clip_test_dataset = SteeringCLIPDataset(
    test_samples,
    clip_processor
)

clip_train_loader = DataLoader(
    clip_train_dataset,
    batch_size=CLIP_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

clip_validation_loader = DataLoader(
    clip_validation_dataset,
    batch_size=CLIP_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

clip_test_loader = DataLoader(
    clip_test_dataset,
    batch_size=CLIP_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

sample_pixels, sample_steering = next(
    iter(clip_train_loader)
)

print("CLIP image batch shape:", sample_pixels.shape)
print("Steering batch shape:", sample_steering.shape)
print("Image dtype:", sample_pixels.dtype)

### Extract CLIP Image Embeddings

Run the frozen encoder without gradients, obtain normalized image embeddings, and concatenate them into NumPy arrays. Feature extraction is performed once so the small regression head can train quickly.


In [ ]:
# ============================================================
# BLOCK 68: EXTRACT CLIP IMAGE EMBEDDINGS
# ============================================================

from tqdm.auto import tqdm

def extract_clip_embeddings(
    data_loader,
    vision_model,
    device
):
    all_embeddings = []
    all_labels = []

    vision_model.eval()

    with torch.no_grad():

        for pixel_values, steering in tqdm(
            data_loader
        ):
            pixel_values = pixel_values.to(
                device,
                non_blocking=True
            )

            outputs = vision_model(
                pixel_values=pixel_values
            )

            embeddings = outputs.image_embeds

            # Normalize CLIP embeddings.
            embeddings = embeddings / embeddings.norm(
                dim=-1,
                keepdim=True
            )

            all_embeddings.append(
                embeddings.cpu().numpy()
            )

            all_labels.append(
                steering.numpy()
            )

    all_embeddings = np.concatenate(
        all_embeddings,
        axis=0
    ).astype(np.float32)

    all_labels = np.concatenate(
        all_labels,
        axis=0
    ).astype(np.float32)

    return all_embeddings, all_labels


embedding_start_time = time.perf_counter()

clip_train_features, clip_train_labels = (
    extract_clip_embeddings(
        clip_train_loader,
        clip_vision_model,
        DEVICE
    )
)

clip_validation_features, clip_validation_labels = (
    extract_clip_embeddings(
        clip_validation_loader,
        clip_vision_model,
        DEVICE
    )
)

clip_test_features, clip_test_labels = (
    extract_clip_embeddings(
        clip_test_loader,
        clip_vision_model,
        DEVICE
    )
)

clip_embedding_time = (
    time.perf_counter() - embedding_start_time
)

print(
    f"\nCLIP embedding extraction time: "
    f"{clip_embedding_time / 60:.2f} minutes"
)

print("Training features:", clip_train_features.shape)
print("Validation features:", clip_validation_features.shape)
print("Test features:", clip_test_features.shape)

print("Training labels:", clip_train_labels.shape)

### Save CLIP Features

Write extracted embeddings and labels to Google Drive. Persisting these arrays avoids repeating the expensive vision-encoder pass after a runtime restart.


In [ ]:
# ============================================================
# BLOCK 69: SAVE CLIP FEATURES
# ============================================================

CLIP_FEATURE_DIR = (
    PROJECT_DIR / "clip_features"
)

CLIP_FEATURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

np.save(
    CLIP_FEATURE_DIR / "train_features.npy",
    clip_train_features
)

np.save(
    CLIP_FEATURE_DIR / "train_labels.npy",
    clip_train_labels
)

np.save(
    CLIP_FEATURE_DIR / "validation_features.npy",
    clip_validation_features
)

np.save(
    CLIP_FEATURE_DIR / "validation_labels.npy",
    clip_validation_labels
)

np.save(
    CLIP_FEATURE_DIR / "test_features.npy",
    clip_test_features
)

np.save(
    CLIP_FEATURE_DIR / "test_labels.npy",
    clip_test_labels
)

print("CLIP embeddings saved to:")
print(CLIP_FEATURE_DIR)

### Define the CLIP Regression Head

Build a TensorFlow multilayer perceptron that maps each 512-dimensional CLIP embedding to one steering value. Batch normalization, dropout, and L2 regularization control overfitting.


In [ ]:
# ============================================================
# BLOCK 70: BUILD CLIP EMBEDDING REGRESSION MODEL
# ============================================================

from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    BatchNormalization
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam

CLIP_FEATURE_DIMENSION = (
    clip_train_features.shape[1]
)

def build_clip_regressor(
    learning_rate=3e-4,
    dropout_rate=0.30,
    l2_strength=1e-4
):
    inputs = Input(
        shape=(CLIP_FEATURE_DIMENSION,),
        name="clip_embedding"
    )

    x = Dense(
        256,
        activation="relu",
        kernel_regularizer=l2(l2_strength),
        name="dense_256"
    )(inputs)

    x = BatchNormalization(
        name="batch_norm_1"
    )(x)

    x = Dropout(
        dropout_rate,
        name="dropout_1"
    )(x)

    x = Dense(
        64,
        activation="relu",
        kernel_regularizer=l2(l2_strength),
        name="dense_64"
    )(x)

    x = Dropout(
        dropout_rate * 0.5,
        name="dropout_2"
    )(x)

    outputs = Dense(
        1,
        activation="linear",
        name="steering_output"
    )(x)

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name="CLIP_Steering_Regressor"
    )

    model.compile(
        optimizer=Adam(
            learning_rate=learning_rate
        ),
        loss="mse",
        metrics=[
            tf.keras.metrics.MeanAbsoluteError(
                name="mae"
            ),
            tf.keras.metrics.RootMeanSquaredError(
                name="rmse"
            )
        ]
    )

    return model


clip_regression_model = build_clip_regressor()

clip_regression_model.summary()

### Train the CLIP Steering Regressor

Train the regression head on precomputed embeddings using MSE loss and validation-based callbacks. Only the small head is optimized; the CLIP encoder remains frozen.


In [ ]:
# ============================================================
# BLOCK 71: TRAIN CLIP STEERING REGRESSION HEAD
# ============================================================

CLIP_REGRESSOR_PATH = (
    MODEL_DIR / "clip_regressor_best.keras"
)

CLIP_REGRESSOR_LOG_PATH = (
    RESULTS_DIR / "clip_regressor_training.csv"
)

clip_callbacks = [
    ModelCheckpoint(
        filepath=str(CLIP_REGRESSOR_PATH),
        monitor="val_mae",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    EarlyStopping(
        monitor="val_mae",
        mode="min",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_mae",
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),

    CSVLogger(
        filename=str(CLIP_REGRESSOR_LOG_PATH),
        append=False
    ),

    TerminateOnNaN()
]

clip_training_start = time.perf_counter()

clip_history = clip_regression_model.fit(
    clip_train_features,
    clip_train_labels,
    validation_data=(
        clip_validation_features,
        clip_validation_labels
    ),
    epochs=40,
    batch_size=64,
    callbacks=clip_callbacks,
    verbose=1
)

clip_regressor_training_time = (
    time.perf_counter() - clip_training_start
)

clip_best_epoch = (
    int(
        np.argmin(
            clip_history.history["val_mae"]
        )
    ) + 1
)

print(
    f"\nCLIP regression training time: "
    f"{clip_regressor_training_time / 60:.2f} minutes"
)

print("Completed epochs:", len(clip_history.history["loss"]))
print("Best epoch:", clip_best_epoch)

print(
    "Best validation MAE:",
    clip_history.history["val_mae"][
        clip_best_epoch - 1
    ]
)

print(
    "Validation RMSE at best epoch:",
    clip_history.history["val_rmse"][
        clip_best_epoch - 1
    ]
)

### Evaluate the Best CLIP Regressor

Reload the best checkpoint and evaluate it on validation and test embeddings.


In [ ]:
# ============================================================
# BLOCK 72: LOAD AND EVALUATE THE BEST CLIP REGRESSOR
# ============================================================

import tensorflow as tf
import pandas as pd
import time

best_clip_regressor = tf.keras.models.load_model(
    str(CLIP_REGRESSOR_PATH)
)

print("Best CLIP regression checkpoint loaded.")
print("Regression-head parameters:", best_clip_regressor.count_params())

clip_validation_results = best_clip_regressor.evaluate(
    clip_validation_features,
    clip_validation_labels,
    verbose=1,
    return_dict=True
)

clip_test_start_time = time.perf_counter()

clip_test_results = best_clip_regressor.evaluate(
    clip_test_features,
    clip_test_labels,
    verbose=1,
    return_dict=True
)

clip_test_evaluation_time = (
    time.perf_counter() - clip_test_start_time
)

clip_evaluation_table = pd.DataFrame(
    [
        {
            "Model": "CLIP vision encoder + regression head",
            "Subset": "Validation",
            "MSE": clip_validation_results["loss"],
            "MAE": clip_validation_results["mae"],
            "RMSE": clip_validation_results["rmse"]
        },
        {
            "Model": "CLIP vision encoder + regression head",
            "Subset": "Test",
            "MSE": clip_test_results["loss"],
            "MAE": clip_test_results["mae"],
            "RMSE": clip_test_results["rmse"]
        }
    ]
)

display(clip_evaluation_table)

print(
    f"CLIP regression test evaluation time: "
    f"{clip_test_evaluation_time:.4f} seconds"
)

### Calculate Complete CLIP Metrics

Generate test predictions and report overall regression metrics, encoder size, trainable head size, feature-extraction time, and head-training time.


In [ ]:
# ============================================================
# BLOCK 73: CALCULATE COMPLETE CLIP TEST METRICS
# ============================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np
import pandas as pd

clip_test_predictions = (
    best_clip_regressor.predict(
        clip_test_features,
        verbose=1
    )
    .reshape(-1)
)

clip_test_actual = (
    clip_test_labels.astype(np.float32)
)

assert len(clip_test_predictions) == len(clip_test_actual)

clip_test_mae = mean_absolute_error(
    clip_test_actual,
    clip_test_predictions
)

clip_test_mse = mean_squared_error(
    clip_test_actual,
    clip_test_predictions
)

clip_test_rmse = np.sqrt(
    clip_test_mse
)

clip_test_r2 = r2_score(
    clip_test_actual,
    clip_test_predictions
)

clip_prediction_bias = np.mean(
    clip_test_predictions - clip_test_actual
)

clip_total_pipeline_time = (
    clip_embedding_time
    + clip_regressor_training_time
)

clip_metrics = pd.DataFrame(
    [
        {
            "Model": "CLIP vision encoder + regression head",
            "Encoder parameters": sum(
                parameter.numel()
                for parameter in clip_vision_model.parameters()
            ),
            "Trainable regression parameters": (
                best_clip_regressor.count_params()
            ),
            "Embedding extraction time (minutes)": (
                clip_embedding_time / 60
            ),
            "Regression training time (minutes)": (
                clip_regressor_training_time / 60
            ),
            "Total experiment time (minutes)": (
                clip_total_pipeline_time / 60
            ),
            "Test MAE": clip_test_mae,
            "Test MSE": clip_test_mse,
            "Test RMSE": clip_test_rmse,
            "Test R2": clip_test_r2,
            "Prediction bias": clip_prediction_bias
        }
    ]
)

display(clip_metrics)

CLIP_METRICS_PATH = (
    RESULTS_DIR / "clip_regressor_metrics.csv"
)

clip_metrics.to_csv(
    CLIP_METRICS_PATH,
    index=False
)

print("CLIP metrics saved to:")
print(CLIP_METRICS_PATH)

### Analyze CLIP by Steering Region

Measure CLIP's error and mean output in each steering category. This is essential for determining whether its low overall MAE is dominated by near-zero predictions.


In [ ]:
# ============================================================
# BLOCK 74: CLIP PERFORMANCE BY STEERING REGION
# ============================================================

clip_prediction_analysis = pd.DataFrame(
    {
        "actual": clip_test_actual,
        "predicted": clip_test_predictions
    }
)

clip_prediction_analysis["absolute_error"] = np.abs(
    clip_prediction_analysis["actual"]
    - clip_prediction_analysis["predicted"]
)

clip_prediction_analysis["steering_region"] = (
    clip_prediction_analysis["actual"]
    .apply(assign_steering_region)
)

clip_region_rows = []

for region in region_order:

    region_data = clip_prediction_analysis[
        clip_prediction_analysis[
            "steering_region"
        ] == region
    ]

    if len(region_data) == 0:
        continue

    clip_region_rows.append(
        {
            "Steering region": region,
            "Test samples": len(region_data),
            "MAE": mean_absolute_error(
                region_data["actual"],
                region_data["predicted"]
            ),
            "RMSE": np.sqrt(
                mean_squared_error(
                    region_data["actual"],
                    region_data["predicted"]
                )
            ),
            "Mean actual": region_data["actual"].mean(),
            "Mean predicted": region_data["predicted"].mean()
        }
    )

clip_region_performance = pd.DataFrame(
    clip_region_rows
)

display(clip_region_performance)

### Compare All Four Model Families

Combine the baseline, improved CNN, MobileNetV2, and CLIP metrics in one table and rank them by test MAE.


In [ ]:
# ============================================================
# BLOCK 75: FINAL COMPARISON OF ALL FOUR MODELS
# ============================================================

all_four_models = pd.DataFrame(
    [
        {
            "Model": "PilotNet baseline",
            "Trainable parameters": best_pilotnet_model.count_params(),
            "Validation MAE": validation_results["mae"],
            "Validation RMSE": validation_results["rmse"],
            "Test MAE": test_mae,
            "Test RMSE": test_rmse,
            "Test R2": test_r2,
            "Prediction bias": test_bias
        },
        {
            "Model": "Improved weighted CNN",
            "Trainable parameters": best_improved_model.count_params(),
            "Validation MAE": validation_results_improved["mae"],
            "Validation RMSE": validation_results_improved["rmse"],
            "Test MAE": improved_test_mae,
            "Test RMSE": improved_test_rmse,
            "Test R2": improved_test_r2,
            "Prediction bias": improved_prediction_bias
        },
        {
            "Model": "Frozen MobileNetV2",
            "Trainable parameters": 168129,
            "Validation MAE": mobilenet_validation_results["mae"],
            "Validation RMSE": mobilenet_validation_results["rmse"],
            "Test MAE": mobilenet_test_mae,
            "Test RMSE": mobilenet_test_rmse,
            "Test R2": mobilenet_test_r2,
            "Prediction bias": mobilenet_prediction_bias
        },
        {
            "Model": "CLIP encoder + regressor",
            "Trainable parameters": best_clip_regressor.count_params(),
            "Validation MAE": clip_validation_results["mae"],
            "Validation RMSE": clip_validation_results["rmse"],
            "Test MAE": clip_test_mae,
            "Test RMSE": clip_test_rmse,
            "Test R2": clip_test_r2,
            "Prediction bias": clip_prediction_bias
        }
    ]
)

all_four_models[
    "Test MAE improvement vs PilotNet (%)"
] = (
    100
    * (
        test_mae
        - all_four_models["Test MAE"]
    )
    / test_mae
)

all_four_models = all_four_models.sort_values(
    "Test MAE"
).reset_index(drop=True)

display(all_four_models)

### Plot Four-Model Metrics

Create final MAE, RMSE, and R-squared charts for the four model families. These metrics reward different aspects of performance and may produce different rankings.


In [ ]:
# ============================================================
# BLOCK 76: FINAL VALIDATION AND TEST METRIC PLOTS
# ============================================================

import matplotlib.pyplot as plt

final_plot_table = all_four_models.set_index(
    "Model"
)

final_plot_table[
    ["Validation MAE", "Test MAE"]
].plot(
    kind="bar",
    figsize=(12, 6)
)

plt.ylabel("Mean absolute error")
plt.title("Validation and Test MAE Across All Models")
plt.xticks(rotation=15)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


final_plot_table[
    ["Validation RMSE", "Test RMSE"]
].plot(
    kind="bar",
    figsize=(12, 6)
)

plt.ylabel("Root mean squared error")
plt.title("Validation and Test RMSE Across All Models")
plt.xticks(rotation=15)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


plt.figure(figsize=(11, 5))

plt.bar(
    final_plot_table.index,
    final_plot_table["Test R2"]
)

plt.axhline(
    0,
    linestyle="--"
)

plt.ylabel("Test R²")
plt.title("Test R² Across All Models")
plt.xticks(rotation=15)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

### Compare Four Models by Steering Region

Add CLIP to the regional comparison and plot category-level MAE for all architectures.


In [ ]:
# ============================================================
# BLOCK 77: FINAL REGION-LEVEL MAE COMPARISON
# ============================================================

clip_region_mae = (
    clip_region_performance[
        ["Steering region", "MAE"]
    ]
    .rename(
        columns={
            "MAE": "CLIP MAE"
        }
    )
)

final_region_comparison = (
    all_region_comparison
    .merge(
        clip_region_mae,
        on="Steering region"
    )
)

final_region_comparison["Steering region"] = pd.Categorical(
    final_region_comparison["Steering region"],
    categories=region_order,
    ordered=True
)

final_region_comparison = (
    final_region_comparison
    .sort_values("Steering region")
    .reset_index(drop=True)
)

display(final_region_comparison)

final_region_comparison.set_index(
    "Steering region"
).plot(
    kind="bar",
    figsize=(13, 6)
)

plt.ylabel("Mean absolute error")
plt.title("Test MAE by Steering Region Across All Models")
plt.xticks(rotation=20)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

### Define Validation Macro-Regional Evaluation

Create a reusable evaluator that calculates ordinary validation metrics plus an equally weighted average of the five steering-region MAEs. This metric reduces the influence of the dominant straight category.


In [ ]:
# ============================================================
# BLOCK 78: VALIDATION METRICS BY STEERING REGION
# ============================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np
import pandas as pd

validation_actual = (
    validation_samples["steering"]
    .to_numpy(dtype=np.float32)
)


def evaluate_validation_regions(
    model,
    dataset,
    actual_values,
    model_name
):
    """
    Evaluate overall and region-level validation performance.
    """

    predictions = (
        model.predict(
            dataset,
            verbose=0
        )
        .reshape(-1)
    )

    overall_mae = mean_absolute_error(
        actual_values,
        predictions
    )

    overall_rmse = np.sqrt(
        mean_squared_error(
            actual_values,
            predictions
        )
    )

    overall_r2 = r2_score(
        actual_values,
        predictions
    )

    analysis = pd.DataFrame(
        {
            "actual": actual_values,
            "predicted": predictions
        }
    )

    analysis["steering_region"] = (
        analysis["actual"]
        .apply(assign_steering_region)
    )

    regional_rows = []

    for region in region_order:

        region_data = analysis[
            analysis["steering_region"] == region
        ]

        if len(region_data) == 0:
            continue

        regional_rows.append(
            {
                "Model": model_name,
                "Steering region": region,
                "Samples": len(region_data),
                "MAE": mean_absolute_error(
                    region_data["actual"],
                    region_data["predicted"]
                ),
                "RMSE": np.sqrt(
                    mean_squared_error(
                        region_data["actual"],
                        region_data["predicted"]
                    )
                )
            }
        )

    regional_table = pd.DataFrame(
        regional_rows
    )

    macro_region_mae = regional_table[
        "MAE"
    ].mean()

    summary = {
        "Model": model_name,
        "Validation MAE": overall_mae,
        "Validation RMSE": overall_rmse,
        "Validation R2": overall_r2,
        "Macro regional MAE": macro_region_mae
    }

    return summary, regional_table, predictions

### Establish the PilotNet Tuning Reference

Evaluate the original PilotNet using the same validation macro-regional metric that will be used to rank tuning trials.


In [ ]:
# ============================================================
# BLOCK 79: BASELINE PILOTNET TUNING REFERENCE
# ============================================================

baseline_validation_summary, \
baseline_validation_regions, \
baseline_validation_predictions = (
    evaluate_validation_regions(
        best_pilotnet_model,
        validation_dataset,
        validation_actual,
        "PilotNet baseline"
    )
)

display(
    pd.DataFrame(
        [baseline_validation_summary]
    )
)

display(
    baseline_validation_regions
)

### Define a Tunable PilotNet

Parameterize learning rate, dropout, L2 strength, and dense-layer width while retaining the core five-convolution PilotNet architecture.


In [ ]:
# ============================================================
# BLOCK 80: DEFINE TUNABLE PILOTNET
# ============================================================

from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    Flatten,
    Dense,
    Dropout
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam

def build_tunable_pilotnet(
    learning_rate=1e-4,
    dropout_rate=0.3,
    l2_strength=0.0,
    dense_width=100
):
    inputs = Input(
        shape=(
            INPUT_HEIGHT,
            INPUT_WIDTH,
            INPUT_CHANNELS
        )
    )

    x = Conv2D(
        24,
        kernel_size=5,
        strides=2,
        activation="elu",
        kernel_regularizer=l2(l2_strength)
    )(inputs)

    x = Conv2D(
        36,
        kernel_size=5,
        strides=2,
        activation="elu",
        kernel_regularizer=l2(l2_strength)
    )(x)

    x = Conv2D(
        48,
        kernel_size=5,
        strides=2,
        activation="elu",
        kernel_regularizer=l2(l2_strength)
    )(x)

    x = Conv2D(
        64,
        kernel_size=3,
        strides=1,
        activation="elu",
        kernel_regularizer=l2(l2_strength)
    )(x)

    x = Conv2D(
        64,
        kernel_size=3,
        strides=1,
        activation="elu",
        kernel_regularizer=l2(l2_strength)
    )(x)

    x = Flatten()(x)

    x = Dense(
        dense_width,
        activation="elu",
        kernel_regularizer=l2(l2_strength)
    )(x)

    x = Dropout(
        dropout_rate
    )(x)

    x = Dense(
        50,
        activation="elu",
        kernel_regularizer=l2(l2_strength)
    )(x)

    x = Dropout(
        dropout_rate * 0.5
    )(x)

    x = Dense(
        10,
        activation="elu",
        kernel_regularizer=l2(l2_strength)
    )(x)

    outputs = Dense(
        1,
        activation="linear"
    )(x)

    model = Model(
        inputs=inputs,
        outputs=outputs
    )

    model.compile(
        optimizer=Adam(
            learning_rate=learning_rate
        ),
        loss="mse",
        metrics=[
            tf.keras.metrics.MeanAbsoluteError(
                name="mae"
            ),
            tf.keras.metrics.RootMeanSquaredError(
                name="rmse"
            )
        ]
    )

    return model

### Run a Compact Hyperparameter Search

Train two additional PilotNet configurations, save their best validation checkpoints, and evaluate overall plus macro-regional validation performance. The test set is not used for tuning.


In [ ]:
# ============================================================
# BLOCK 81: COMPACT PILOTNET HYPERPARAMETER SEARCH
# ============================================================

import gc
import time
import numpy as np
import pandas as pd
import tensorflow as tf

tuning_configs = [
    {
        "name": "PilotNet_low_dropout",
        "learning_rate": 3e-4,
        "dropout_rate": 0.25,
        "l2_strength": 1e-5,
        "dense_width": 100
    },
    {
        "name": "PilotNet_regularized",
        "learning_rate": 1e-4,
        "dropout_rate": 0.20,
        "l2_strength": 1e-4,
        "dense_width": 128
    }
]

tuning_results = [
    baseline_validation_summary
]

tuning_region_tables = {
    "PilotNet baseline": baseline_validation_regions
}

tuned_model_paths = {}

for config in tuning_configs:

    print("\n" + "=" * 70)
    print("Training:", config["name"])
    print(config)
    print("=" * 70)

    tf.keras.backend.clear_session()
    gc.collect()

    model = build_tunable_pilotnet(
        learning_rate=config["learning_rate"],
        dropout_rate=config["dropout_rate"],
        l2_strength=config["l2_strength"],
        dense_width=config["dense_width"]
    )

    checkpoint_path = (
        MODEL_DIR
        / f"{config['name']}_best.keras"
    )

    callbacks = [
        ModelCheckpoint(
            filepath=str(checkpoint_path),
            monitor="val_mae",
            mode="min",
            save_best_only=True,
            verbose=0
        ),

        EarlyStopping(
            monitor="val_mae",
            mode="min",
            patience=4,
            restore_best_weights=True,
            verbose=1
        ),

        ReduceLROnPlateau(
            monitor="val_mae",
            mode="min",
            factor=0.5,
            patience=2,
            min_lr=1e-7,
            verbose=1
        ),

        TerminateOnNaN()
    ]

    start_time = time.perf_counter()

    history = model.fit(
        train_dataset,
        validation_data=validation_dataset,
        epochs=15,
        callbacks=callbacks,
        verbose=1
    )

    elapsed_minutes = (
        time.perf_counter() - start_time
    ) / 60

    best_model = tf.keras.models.load_model(
        str(checkpoint_path)
    )

    summary, regional_table, predictions = (
        evaluate_validation_regions(
            best_model,
            validation_dataset,
            validation_actual,
            config["name"]
        )
    )

    summary["Training time (minutes)"] = elapsed_minutes
    summary["Parameters"] = best_model.count_params()
    summary["Learning rate"] = config["learning_rate"]
    summary["Dropout"] = config["dropout_rate"]
    summary["L2"] = config["l2_strength"]
    summary["Dense width"] = config["dense_width"]

    tuning_results.append(summary)

    tuning_region_tables[
        config["name"]
    ] = regional_table

    tuned_model_paths[
        config["name"]
    ] = checkpoint_path

tuning_results_table = pd.DataFrame(
    tuning_results
)

display(
    tuning_results_table.sort_values(
        [
            "Macro regional MAE",
            "Validation MAE"
        ]
    ).reset_index(drop=True)
)

### Select the Best Tuned PilotNet

Rank tuning candidates primarily by macro-regional validation MAE and secondarily by ordinary validation MAE, then load the selected checkpoint.


In [ ]:
# ============================================================
# BLOCK 82: SELECT BEST TUNED PILOTNET
# ============================================================

ranked_tuning_results = (
    tuning_results_table
    .sort_values(
        [
            "Macro regional MAE",
            "Validation MAE"
        ]
    )
    .reset_index(drop=True)
)

display(ranked_tuning_results)

best_tuned_name = ranked_tuning_results.loc[
    0,
    "Model"
]

print("Selected tuning result:")
print(best_tuned_name)

if best_tuned_name == "PilotNet baseline":

    best_tuned_pilotnet = best_pilotnet_model
    best_tuned_pilotnet_path = PILOTNET_MODEL_PATH

else:

    best_tuned_pilotnet_path = (
        tuned_model_paths[best_tuned_name]
    )

    best_tuned_pilotnet = (
        tf.keras.models.load_model(
            str(best_tuned_pilotnet_path)
        )
    )

print("Selected checkpoint:")
print(best_tuned_pilotnet_path)

print("\nSelected validation regional results:")
display(
    tuning_region_tables[
        best_tuned_name
    ]
)

### Evaluate the Selected Tuned PilotNet

Run the chosen configuration on the untouched test dataset and record Keras metrics and evaluation time.


In [ ]:
# ============================================================
# BLOCK 83: EVALUATE SELECTED TUNED PILOTNET ON TEST SET
# ============================================================

import time
import numpy as np
import pandas as pd

tuned_test_start_time = time.perf_counter()

tuned_test_results = best_tuned_pilotnet.evaluate(
    test_dataset,
    verbose=1,
    return_dict=True
)

tuned_test_evaluation_time = (
    time.perf_counter() - tuned_test_start_time
)

print("Selected tuned model:", best_tuned_name)
print("\nKeras test results:")
print(tuned_test_results)

print(
    f"\nTest evaluation time: "
    f"{tuned_test_evaluation_time:.2f} seconds"
)

### Calculate Tuned-PilotNet Test Metrics

Generate predictions and compute the complete set of overall test metrics for the selected tuned configuration.


In [ ]:
# ============================================================
# BLOCK 84: CALCULATE COMPLETE TUNED PILOTNET TEST METRICS
# ============================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

tuned_test_predictions = (
    best_tuned_pilotnet.predict(
        test_dataset,
        verbose=1
    )
    .reshape(-1)
)

assert len(tuned_test_predictions) == len(test_actual)

tuned_test_mae = mean_absolute_error(
    test_actual,
    tuned_test_predictions
)

tuned_test_mse = mean_squared_error(
    test_actual,
    tuned_test_predictions
)

tuned_test_rmse = np.sqrt(
    tuned_test_mse
)

tuned_test_r2 = r2_score(
    test_actual,
    tuned_test_predictions
)

tuned_prediction_bias = np.mean(
    tuned_test_predictions - test_actual
)

tuned_pilotnet_metrics = pd.DataFrame(
    [
        {
            "Model": best_tuned_name,
            "Parameters": best_tuned_pilotnet.count_params(),
            "Test MAE": tuned_test_mae,
            "Test MSE": tuned_test_mse,
            "Test RMSE": tuned_test_rmse,
            "Test R2": tuned_test_r2,
            "Prediction bias": tuned_prediction_bias
        }
    ]
)

display(tuned_pilotnet_metrics)

TUNED_PILOTNET_METRICS_PATH = (
    RESULTS_DIR / "tuned_pilotnet_metrics.csv"
)

tuned_pilotnet_metrics.to_csv(
    TUNED_PILOTNET_METRICS_PATH,
    index=False
)

print("Metrics saved to:")
print(TUNED_PILOTNET_METRICS_PATH)

### Evaluate Tuned PilotNet by Steering Region

Calculate category-specific errors and the final macro-regional test MAE. This determines whether validation-based balanced tuning generalized to unseen data.


In [ ]:
# ============================================================
# BLOCK 85: TUNED PILOTNET TEST PERFORMANCE BY REGION
# ============================================================

tuned_prediction_analysis = pd.DataFrame(
    {
        "actual": test_actual,
        "predicted": tuned_test_predictions
    }
)

tuned_prediction_analysis["absolute_error"] = np.abs(
    tuned_prediction_analysis["actual"]
    - tuned_prediction_analysis["predicted"]
)

tuned_prediction_analysis["steering_region"] = (
    tuned_prediction_analysis["actual"]
    .apply(assign_steering_region)
)

tuned_region_rows = []

for region in region_order:

    region_data = tuned_prediction_analysis[
        tuned_prediction_analysis[
            "steering_region"
        ] == region
    ]

    if len(region_data) == 0:
        continue

    tuned_region_rows.append(
        {
            "Steering region": region,
            "Test samples": len(region_data),
            "MAE": mean_absolute_error(
                region_data["actual"],
                region_data["predicted"]
            ),
            "RMSE": np.sqrt(
                mean_squared_error(
                    region_data["actual"],
                    region_data["predicted"]
                )
            ),
            "Mean actual": region_data["actual"].mean(),
            "Mean predicted": region_data["predicted"].mean()
        }
    )

tuned_region_performance = pd.DataFrame(
    tuned_region_rows
)

display(tuned_region_performance)

tuned_macro_region_mae = (
    tuned_region_performance["MAE"].mean()
)

print(
    "Tuned PilotNet macro regional test MAE:",
    tuned_macro_region_mae
)

### Compare Original and Tuned PilotNet

Place overall and macro-regional metrics side by side and calculate percentage improvements. This exposes the trade-off between average accuracy and turn balance.


In [ ]:
# ============================================================
# BLOCK 86: ORIGINAL VERSUS TUNED PILOTNET
# ============================================================

pilotnet_tuning_test_comparison = pd.DataFrame(
    [
        {
            "Model": "Original PilotNet",
            "Parameters": best_pilotnet_model.count_params(),
            "Test MAE": test_mae,
            "Test RMSE": test_rmse,
            "Test R2": test_r2,
            "Prediction bias": test_bias,
            "Macro regional test MAE": (
                region_performance["MAE"].mean()
            )
        },
        {
            "Model": best_tuned_name,
            "Parameters": best_tuned_pilotnet.count_params(),
            "Test MAE": tuned_test_mae,
            "Test RMSE": tuned_test_rmse,
            "Test R2": tuned_test_r2,
            "Prediction bias": tuned_prediction_bias,
            "Macro regional test MAE": (
                tuned_macro_region_mae
            )
        }
    ]
)

pilotnet_tuning_test_comparison[
    "Overall MAE improvement (%)"
] = (
    100
    * (
        test_mae
        - pilotnet_tuning_test_comparison["Test MAE"]
    )
    / test_mae
)

pilotnet_tuning_test_comparison[
    "Macro regional MAE improvement (%)"
] = (
    100
    * (
        region_performance["MAE"].mean()
        - pilotnet_tuning_test_comparison[
            "Macro regional test MAE"
        ]
    )
    / region_performance["MAE"].mean()
)

display(pilotnet_tuning_test_comparison)

### Compare PilotNet Variants by Region

Merge original and tuned region tables and quantify the improvement or deterioration separately for each maneuver.


In [ ]:
# ============================================================
# BLOCK 87: ORIGINAL VERSUS TUNED PILOTNET BY REGION
# ============================================================

original_pilotnet_regions = (
    region_performance[
        [
            "Steering region",
            "MAE",
            "RMSE",
            "Mean predicted"
        ]
    ]
    .rename(
        columns={
            "MAE": "Original MAE",
            "RMSE": "Original RMSE",
            "Mean predicted": "Original mean prediction"
        }
    )
)

tuned_pilotnet_regions = (
    tuned_region_performance[
        [
            "Steering region",
            "MAE",
            "RMSE",
            "Mean predicted"
        ]
    ]
    .rename(
        columns={
            "MAE": "Tuned MAE",
            "RMSE": "Tuned RMSE",
            "Mean predicted": "Tuned mean prediction"
        }
    )
)

pilotnet_region_comparison = (
    original_pilotnet_regions.merge(
        tuned_pilotnet_regions,
        on="Steering region"
    )
)

pilotnet_region_comparison[
    "MAE improvement (%)"
] = (
    100
    * (
        pilotnet_region_comparison["Original MAE"]
        - pilotnet_region_comparison["Tuned MAE"]
    )
    / pilotnet_region_comparison["Original MAE"]
)

display(pilotnet_region_comparison)

## Results Summary and Discussion

The models were evaluated using ordinary regression metrics and steering-region metrics. Overall MAE is useful, but it is strongly affected by the large number of straight-driving samples. Macro regional MAE is therefore used as a complementary metric because it gives equal importance to sharp-left, left, straight, right, and sharp-right cases.

| Model | Test MAE | Test RMSE | Test R² | Macro regional MAE | Main interpretation |
|---|---:|---:|---:|---:|---|
| Original PilotNet | 0.09620 | 0.13646 | 0.14369 | 0.18040 | Best RMSE and R² among the original models; balanced but underpredicts sharp turns |
| Tuned PilotNet | 0.10340 | 0.13850 | 0.11782 | 0.15440 | Best balanced regional performance and improved sharp-turn prediction |
| Improved weighted CNN | 0.09376 | 0.15034 | -0.03945 | 0.22774 | Compact but weaker squared-error and regional behavior |
| Frozen MobileNetV2 | 0.07657 | 0.14048 | 0.09251 | 0.22222 | Strong transfer-learning MAE but weak sharp-turn magnitude |
| CLIP encoder + regressor | 0.07373 | 0.14526 | 0.02968 | 0.23419 | Best overall MAE due to excellent straight-driving prediction, but poor sharp-turn behavior |

The results show that the lowest overall MAE does not necessarily correspond to the safest or most balanced steering behavior. CLIP achieved the lowest average error because it performed extremely well on straight-driving frames, which dominate the test set. However, the tuned PilotNet achieved the best macro regional MAE and the strongest sharp-turn performance. This makes the tuned PilotNet the most behaviorally balanced model, even though its ordinary MAE is not the lowest.

### Assemble the Final Five-Model Comparison

Add tuned PilotNet to the conventional, transfer-learning, and CLIP models and provide rankings by both overall MAE and macro-regional MAE.


In [ ]:
# ============================================================
# BLOCK 88: FINAL COMPARISON INCLUDING TUNED PILOTNET
# ============================================================

final_model_comparison = pd.DataFrame(
    [
        {
            "Model": "Original PilotNet",
            "Trainable parameters": best_pilotnet_model.count_params(),
            "Test MAE": test_mae,
            "Test RMSE": test_rmse,
            "Test R2": test_r2,
            "Prediction bias": test_bias,
            "Macro regional MAE": (
                region_performance["MAE"].mean()
            )
        },
        {
            "Model": best_tuned_name,
            "Trainable parameters": best_tuned_pilotnet.count_params(),
            "Test MAE": tuned_test_mae,
            "Test RMSE": tuned_test_rmse,
            "Test R2": tuned_test_r2,
            "Prediction bias": tuned_prediction_bias,
            "Macro regional MAE": tuned_macro_region_mae
        },
        {
            "Model": "Improved weighted CNN",
            "Trainable parameters": best_improved_model.count_params(),
            "Test MAE": improved_test_mae,
            "Test RMSE": improved_test_rmse,
            "Test R2": improved_test_r2,
            "Prediction bias": improved_prediction_bias,
            "Macro regional MAE": (
                improved_region_performance["MAE"].mean()
            )
        },
        {
            "Model": "Frozen MobileNetV2",
            "Trainable parameters": 168129,
            "Test MAE": mobilenet_test_mae,
            "Test RMSE": mobilenet_test_rmse,
            "Test R2": mobilenet_test_r2,
            "Prediction bias": mobilenet_prediction_bias,
            "Macro regional MAE": (
                mobilenet_region_performance["MAE"].mean()
            )
        },
        {
            "Model": "CLIP encoder + regressor",
            "Trainable parameters": best_clip_regressor.count_params(),
            "Test MAE": clip_test_mae,
            "Test RMSE": clip_test_rmse,
            "Test R2": clip_test_r2,
            "Prediction bias": clip_prediction_bias,
            "Macro regional MAE": (
                clip_region_performance["MAE"].mean()
            )
        }
    ]
)

display(
    final_model_comparison.sort_values(
        "Test MAE"
    ).reset_index(drop=True)
)

print("\nRanking by balanced regional performance:")

display(
    final_model_comparison.sort_values(
        "Macro regional MAE"
    ).reset_index(drop=True)
)

### Plot Final Model Rankings

Visualize overall test error, balanced regional error, and explained variance for all models.


In [ ]:
# ============================================================
# BLOCK 89: FINAL MODEL-COMPARISON PLOTS
# ============================================================

import matplotlib.pyplot as plt

final_plot_data = (
    final_model_comparison.set_index("Model")
)

final_plot_data[
    ["Test MAE", "Test RMSE"]
].plot(
    kind="bar",
    figsize=(13, 6)
)

plt.ylabel("Error")
plt.title("Overall Test Error Across Models")
plt.xticks(rotation=20)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


plt.figure(figsize=(12, 5))

plt.bar(
    final_plot_data.index,
    final_plot_data["Macro regional MAE"]
)

plt.ylabel("Macro regional MAE")
plt.title(
    "Balanced Steering-Region Performance Across Models"
)

plt.xticks(rotation=20)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


plt.figure(figsize=(12, 5))

plt.bar(
    final_plot_data.index,
    final_plot_data["Test R2"]
)

plt.axhline(
    0,
    linestyle="--"
)

plt.ylabel("Test R²")
plt.title("Explained Steering Variance Across Models")
plt.xticks(rotation=20)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

### Create the Definitive Results Table

Build a detailed final table containing model type, parameters, overall metrics, macro-regional MAE, and sharp-turn errors. Save the table for reporting and presentation use.


In [ ]:
# ============================================================
# BLOCK 90: DEFINITIVE FINAL MODEL COMPARISON
# ============================================================

import pandas as pd
import numpy as np

final_results = pd.DataFrame(
    [
        {
            "Model": "Original PilotNet",
            "Model type": "CNN trained from scratch",
            "Trainable parameters": 252219,
            "Test MAE": test_mae,
            "Test RMSE": test_rmse,
            "Test R2": test_r2,
            "Prediction bias": test_bias,
            "Macro regional MAE": region_performance["MAE"].mean(),
            "Sharp-left MAE": region_performance.loc[
                region_performance["Steering region"] == "Sharp left",
                "MAE"
            ].iloc[0],
            "Sharp-right MAE": region_performance.loc[
                region_performance["Steering region"] == "Sharp right",
                "MAE"
            ].iloc[0]
        },
        {
            "Model": "Tuned PilotNet",
            "Model type": "Hyperparameter-tuned CNN",
            "Trainable parameters": best_tuned_pilotnet.count_params(),
            "Test MAE": tuned_test_mae,
            "Test RMSE": tuned_test_rmse,
            "Test R2": tuned_test_r2,
            "Prediction bias": tuned_prediction_bias,
            "Macro regional MAE": tuned_macro_region_mae,
            "Sharp-left MAE": tuned_region_performance.loc[
                tuned_region_performance["Steering region"] == "Sharp left",
                "MAE"
            ].iloc[0],
            "Sharp-right MAE": tuned_region_performance.loc[
                tuned_region_performance["Steering region"] == "Sharp right",
                "MAE"
            ].iloc[0]
        },
        {
            "Model": "Improved weighted CNN",
            "Model type": "Regularized weighted CNN",
            "Trainable parameters": 137273,
            "Test MAE": improved_test_mae,
            "Test RMSE": improved_test_rmse,
            "Test R2": improved_test_r2,
            "Prediction bias": improved_prediction_bias,
            "Macro regional MAE": improved_region_performance["MAE"].mean(),
            "Sharp-left MAE": improved_region_performance.loc[
                improved_region_performance["Steering region"] == "Sharp left",
                "MAE"
            ].iloc[0],
            "Sharp-right MAE": improved_region_performance.loc[
                improved_region_performance["Steering region"] == "Sharp right",
                "MAE"
            ].iloc[0]
        },
        {
            "Model": "Frozen MobileNetV2",
            "Model type": "ImageNet transfer learning",
            "Trainable parameters": 168129,
            "Test MAE": mobilenet_test_mae,
            "Test RMSE": mobilenet_test_rmse,
            "Test R2": mobilenet_test_r2,
            "Prediction bias": mobilenet_prediction_bias,
            "Macro regional MAE": mobilenet_region_performance["MAE"].mean(),
            "Sharp-left MAE": mobilenet_region_performance.loc[
                mobilenet_region_performance["Steering region"] == "Sharp left",
                "MAE"
            ].iloc[0],
            "Sharp-right MAE": mobilenet_region_performance.loc[
                mobilenet_region_performance["Steering region"] == "Sharp right",
                "MAE"
            ].iloc[0]
        },
        {
            "Model": "CLIP encoder + regressor",
            "Model type": "Vision-language-pretrained encoder",
            "Trainable parameters": 148865,
            "Test MAE": clip_test_mae,
            "Test RMSE": clip_test_rmse,
            "Test R2": clip_test_r2,
            "Prediction bias": clip_prediction_bias,
            "Macro regional MAE": clip_region_performance["MAE"].mean(),
            "Sharp-left MAE": clip_region_performance.loc[
                clip_region_performance["Steering region"] == "Sharp left",
                "MAE"
            ].iloc[0],
            "Sharp-right MAE": clip_region_performance.loc[
                clip_region_performance["Steering region"] == "Sharp right",
                "MAE"
            ].iloc[0]
        }
    ]
)

display(
    final_results.sort_values(
        "Test MAE"
    ).reset_index(drop=True)
)

print("\nRanking by balanced regional performance:")

display(
    final_results.sort_values(
        "Macro regional MAE"
    ).reset_index(drop=True)
)

FINAL_RESULTS_PATH = (
    RESULTS_DIR / "final_model_comparison.csv"
)

final_results.to_csv(
    FINAL_RESULTS_PATH,
    index=False
)

print("\nFinal comparison saved to:")
print(FINAL_RESULTS_PATH)

### Contrast Overall and Balanced Accuracy

Plot test MAE beside macro-regional MAE. The difference between these bars demonstrates how an imbalanced test distribution can favor models that predict straight driving well.


In [ ]:
# ============================================================
# BLOCK 91: PLOT OVERALL AND BALANCED MODEL PERFORMANCE
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

plot_results = final_results.set_index("Model")

x = np.arange(len(plot_results))
bar_width = 0.36

plt.figure(figsize=(13, 6))

plt.bar(
    x - bar_width / 2,
    plot_results["Test MAE"],
    width=bar_width,
    label="Overall test MAE"
)

plt.bar(
    x + bar_width / 2,
    plot_results["Macro regional MAE"],
    width=bar_width,
    label="Macro regional MAE"
)

plt.xticks(
    x,
    plot_results.index,
    rotation=20
)

plt.ylabel("Mean absolute error")
plt.title(
    "Overall Accuracy versus Balanced Steering Performance"
)

plt.legend()
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

### Export Trained Models to HDF5

Load saved Keras checkpoints and export available models to legacy `.h5` files for local deployment. Missing checkpoints are skipped and file sizes are recorded.


In [ ]:
# ============================================================
# BLOCK 92: EXPORT TRAINED MODELS TO HDF5 (.h5)
# ============================================================

from pathlib import Path
import tensorflow as tf
import pandas as pd

MODEL_DIR = Path(
    "/content/drive/MyDrive/ML_Python_Project/trained_models"
)

H5_EXPORT_DIR = MODEL_DIR / "h5_exports"

H5_EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Map each saved .keras checkpoint to its desired .h5 filename.
model_exports = {
    "pilotnet_baseline_best.keras":
        "pilotnet_baseline.h5",

    "improved_weighted_cnn_best.keras":
        "improved_weighted_cnn.h5",

    "mobilenetv2_frozen_best.keras":
        "mobilenetv2_frozen.h5",

    "mobilenetv2_finetuned_best.keras":
        "mobilenetv2_finetuned.h5",

    "clip_regressor_best.keras":
        "clip_regression_head.h5",

    "PilotNet_low_dropout_best.keras":
        "pilotnet_tuned_low_dropout.h5",

    "PilotNet_regularized_best.keras":
        "pilotnet_regularized.h5"
}

export_results = []

for source_name, destination_name in model_exports.items():

    source_path = MODEL_DIR / source_name
    destination_path = H5_EXPORT_DIR / destination_name

    if not source_path.exists():

        print(f"Skipped: {source_name} was not found.")

        export_results.append(
            {
                "Source model": source_name,
                "Exported file": destination_name,
                "Status": "Not found"
            }
        )

        continue

    print(f"\nLoading: {source_path}")

    try:
        model = tf.keras.models.load_model(
            source_path,
            compile=False
        )

        # Saving with an .h5 extension uses the legacy HDF5 format.
        model.save(
            destination_path,
            include_optimizer=False
        )

        file_size_mb = (
            destination_path.stat().st_size
            / (1024 ** 2)
        )

        print(f"Saved: {destination_path}")
        print(f"Size: {file_size_mb:.2f} MB")

        export_results.append(
            {
                "Source model": source_name,
                "Exported file": destination_name,
                "Status": "Exported",
                "Size (MB)": file_size_mb
            }
        )

    except Exception as error:

        print(f"Failed to export {source_name}")
        print(error)

        export_results.append(
            {
                "Source model": source_name,
                "Exported file": destination_name,
                "Status": f"Failed: {error}"
            }
        )

export_results_table = pd.DataFrame(
    export_results
)

display(export_results_table)

print("\nH5 files are stored in:")
print(H5_EXPORT_DIR)

### Verify Exported HDF5 Models

Reload every exported `.h5` file without compilation and report parameter counts and input/output shapes. Successful reloading confirms that the deployment files are structurally valid.


In [ ]:
# ============================================================
# BLOCK 93: VERIFY EXPORTED H5 MODELS
# ============================================================

verification_results = []

h5_files = sorted(
    H5_EXPORT_DIR.glob("*.h5")
)

print("H5 files found:", len(h5_files))

for h5_path in h5_files:

    try:
        model = tf.keras.models.load_model(
            h5_path,
            compile=False
        )

        verification_results.append(
            {
                "File": h5_path.name,
                "Status": "Loaded successfully",
                "Parameters": model.count_params(),
                "Input shape": str(model.input_shape),
                "Output shape": str(model.output_shape)
            }
        )

        print(
            f"Verified: {h5_path.name} "
            f"({model.count_params():,} parameters)"
        )

    except Exception as error:

        verification_results.append(
            {
                "File": h5_path.name,
                "Status": f"Failed: {error}"
            }
        )

verification_table = pd.DataFrame(
    verification_results
)

display(verification_table)

## Limitations and Future Work

This project demonstrates image-based steering prediction in a simulated environment, but several limitations remain:

- The dataset is synthetic and generated from the Udacity simulator, so results may not transfer directly to real roads.
- The data are strongly imbalanced toward straight driving, making ordinary MAE potentially misleading.
- The models are single-frame predictors and do not explicitly use temporal history.
- The throttle command used during deployment is a simple speed controller rather than a learned control policy.
- Offline test metrics do not fully predict closed-loop driving behavior because errors can accumulate over time in the simulator.
- The CLIP experiment uses a frozen vision-language-pretrained visual encoder, but it is not a full vision-language-action model.

Future improvements could include collecting more recovery and sharp-turn data, training sequence-based models such as CNN-LSTM or video transformers, predicting steering and throttle jointly, evaluating multiple tracks, adding closed-loop lap-success metrics, and testing robustness under different lighting or road conditions.

### Next Steps

Using the models (one by one) in the drive_simulator.py script (provided with the submission package), run the inference on the simulator to visualize performance of the models. A sample model in autonomous driving mode predicting the steering commands is shown in the video below: 
 <video controls src="ML_Python-1.mp4" title="Title"></video>

## Submission Note

This notebook can be submitted directly as the project document in `.ipynb` format. It includes project motivation, team role, design framework, model architectures, methods, setup instructions, visual evidence of execution, model results, discussion, limitations, and future work. A short demo video of the simulator is optional and can be submitted separately if required.